# Предсказание следующего продукта с помощью GRU

В этом ноутбуке решается задача предсказания следующего продукта, с которым взаимодействует клиент банка, на основе истории его действий

Используется гибридная нейронная архитектура, объединяющая:

- **GRU (Gated Recurrent Unit)** - анализирует последовательность последних 512 событий с учётом их порядка и временных характеристик
- **Attention pooling** - выделяет наиболее информативные скрытые состояния GRU и формирует представление последовательности
- **MLP (Multilayer Perceptron)** - обрабатывает агрегированные признаки всей доступной истории клиента и его взаимодействий с продуктами

Полученные представления объединяются для предсказания одного из 32 целевых продуктов.

**Результаты на Kaggle Public Leaderboard:**

| Модель | Accuracy |
|---|---:|
| GRU | **0.41203** |

Лучший результат получен самостоятельной GRU

## 1. Данные и настройки

Подключим библиотеки, зададим параметры модели и обучения, определим устройство (CPU/GPU)

Загрузим метаданные обучающей и тестовой выборок, а также словарь событий. Подготовим индексы категориальных признаков для embedding-слоёв нейросети

In [ ]:

from pathlib import Path
from time import perf_counter
import hashlib, json, math, os, random, shutil

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from IPython.display import display


# параметры подготовки данных, архитектуры GRU и обучения
CFG = dict(
    seed=42,
    max_len=512,        # 512 последних событий
    arrow_batch=64,
    batch_size=128,     # примеров в одном батче
    hidden=128,         # скрытое состояние gru
    layers=2,           # 2 слоя
    dropout=0.20,
    epochs=15,
    patience=3,
    lr=0.001,
    weight_decay=0.0001,
    grad_clip=1.0,
    activity_top=16
)

PREPROCESS_VERSION = 'gru_full_history_v1_1'
SMOKE_TEST = False  # быстрая проверка перел обучением

ON_KAGGLE = Path('/kaggle/input').is_dir()
required = [
    'train.parquet',
    'test.parquet',
    'event_dictionary.parquet',
    'sample_submission.csv'
]

if ON_KAGGLE:
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_DIR = next((path for base in [Path.cwd(), *Path.cwd().parents]
                        for path in [base, base / 'task3']
                        if all((path / 'data' / name).is_file() for name in required)), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Нужны четыре исходных файла соревнования в task3/data/.')
    INPUT_ROOT = PROJECT_DIR

folders = sorted({
    p.parent for p in INPUT_ROOT.rglob('train.parquet')
    if all((p.parent / name).is_file() for name in required)
})

if len(folders) != 1:
    raise ValueError(f'Нужен один комплект исходных данных; найдено: {folders}')

DATA_DIR = folders[0]


# Сохраняем промежуточные результаты отдельно от исходных данных.
WORK = (
    Path('/kaggle/working') if ON_KAGGLE
    else Path('/tmp')
) / ('gru_smoke' if SMOKE_TEST else 'gru_v1')

WORK.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR = Path('/kaggle/working') if ON_KAGGLE else PROJECT_DIR / 'submits'
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


# Используем GPU при наличии; mixed precision включаем только на CUDA.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'


# фиксируем настройки вычислений и генераторы случайных чисел
torch.set_num_threads(2)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])


print('Устройство:', DEVICE, '| PyTorch:', torch.__version__)
if DEVICE.type == 'cpu' and not SMOKE_TEST:
    print('Для полного обучения включите GPU. Подготовка данных работает на CPU.')


In [ ]:
# вычисляем хеши файлов 
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

source_hashes = {name: sha256_file(DATA_DIR / name) for name in required}


# Загружаем метаданные без длинных последовательностей событий.
train_meta = pq.read_table(
    DATA_DIR / 'train.parquet',
    columns=['example_id', 'client_id', 'target_product']
).to_pandas()

test_meta = pq.read_table(
    DATA_DIR / 'test.parquet',
    columns=['example_id', 'client_id']
).to_pandas()

dictionary = pq.read_table(
    DATA_DIR / 'event_dictionary.parquet'
).to_pandas()


# формируем 32 целевых класса и переводим ответы в индексы от 0 до 31
classes = np.sort(train_meta.target_product.unique()).astype(np.int32)
assert len(classes) == 32

if SMOKE_TEST:
    train_meta, test_meta = train_meta.iloc[:64].copy(), test_meta.iloc[:16].copy()
    CFG.update(max_len=64, batch_size=8, epochs=1)

y = np.searchsorted(classes, train_meta.target_product.to_numpy()).astype(np.int64)

assert train_meta.example_id.is_unique and test_meta.example_id.is_unique
assert dictionary.encoded_event.is_unique


# оставляем первые индексы для embedding:
# события: 0 - пустая позиция, 1 - неизвестное событие;
# продукты: 0 - пустая позиция, 1 - нет продукта, 2 - неизвестный продукт.
event_codes = sorted(dictionary.encoded_event.astype(int))
event_map = {value: i + 2 for i, value in enumerate(event_codes)}

type_map = {
    'view': 1,
    'click': 2,
    'other': 3,
    'session_boundary': 4
}


# таблицы типов событий и признака is_target_event
event_types = np.full(len(event_map) + 2, 5, dtype=np.int64)
event_types[0] = 0

event_targets = np.zeros(len(event_map) + 2, dtype=np.float32)

for row in dictionary.itertuples(index=False):
    event_types[event_map[int(row.encoded_event)]] = type_map.get(
        row.event_type, 5
    )
    event_targets[event_map[int(row.encoded_event)]] = float(
        row.is_target_event
    )

product_map = {int(value): i + 3 for i, value in enumerate(classes)}

print(f'Train: {len(train_meta):,}; test: {len(test_meta):,}; классов: {len(classes)}')


## 2. Разбиение данных

Разделим обучающую выборку с помощью **StratifiedGroupKFold**

Валидационную часть дополнительно разделим на две:

- **ES (Early Stopping)**  для ранней остановки обучения
- **Outer validation** - для итоговой оценки модели на отложенных данных

Разбиение проверено по ранее сохранённым индексам: 79 999 fit / 5 000 ES / 15 001 outer. Контрольные суммы ниже защищают от незаметного изменения индексов между версиями библиотек. Checkpoint успешного GPU-запуска при этой проверке не был доступен, поэтому совпадение индексов само по себе не доказывает точное воспроизведение Public Score 0.41203.


In [ ]:
train_idx, valid_idx = next(
    StratifiedGroupKFold(5, shuffle=True, random_state=42)
    .split(train_meta, y, train_meta.client_id)
)
train_idx, valid_idx = np.sort(train_idx), np.sort(valid_idx)

# для early stopping оставляем только классы, присутствующие в обучении
known_valid = valid_idx[np.isin(y[valid_idx], np.unique(y[train_idx]))]

# выбираем клиентов для ES
clients = np.unique(train_meta.iloc[known_valid].client_id)
es_clients = np.random.default_rng(2068).choice(
    clients,
    min(5000, max(1, len(clients) // 4)),
    replace=False
)

es_idx = known_valid[
    train_meta.iloc[known_valid].client_id.isin(es_clients).to_numpy()
]
outer_idx = valid_idx[
    ~train_meta.iloc[valid_idx].client_id.isin(es_clients).to_numpy()
]

# проверяем полноту разбиения и отсутствие пересечений клиентов
assert np.array_equal(
    np.sort(np.concatenate([train_idx, valid_idx])),
    np.arange(len(train_meta))
)
assert np.array_equal(
    np.sort(np.concatenate([es_idx, outer_idx])),
    valid_idx
)

for a, b in [(train_idx, valid_idx), (es_idx, outer_idx)]:
    assert not set(train_meta.iloc[a].client_id) & set(
        train_meta.iloc[b].client_id
    )

assert len(es_idx) and len(outer_idx)

print('Fit / ES / outer:', len(train_idx), len(es_idx), len(outer_idx))
print(
    'Классов в fit / ES / outer:',
    *(len(np.unique(y[idx])) for idx in [train_idx, es_idx, outer_idx])
)

# Проверяем сохранённое разбиение без внешних файлов.
if not SMOKE_TEST:
    expected_split_hashes = {'train_idx': '2c69ddc930eaaed6bfe78c9ebc82891b68eddf84984a5fa07eccdcf84db2fed6', 'valid_idx': 'adcb3065fa7e1b5f659efff8dae41cf802d81b781dae8959b9c8c9e4842e2e21', 'es_idx': '485069eeb1ff71519991dcd0c35f809b5d157a22990e78234732ad8a0c981750', 'outer_idx': 'a24fb981a6c4f1964b3741cb9845d61295345a1c2f9e9b553748cd1a8057b4c9'}
    for name, rows in [('train_idx', train_idx), ('valid_idx', valid_idx),
                       ('es_idx', es_idx), ('outer_idx', outer_idx)]:
        digest = hashlib.sha256(np.asarray(rows, dtype='<i8').tobytes()).hexdigest()
        assert digest == expected_split_hashes[name], (
            'Разбиение отличается от проверенного: ' + name
            + '. Проверьте данные и версию scikit-learn; индексы не подменяются автоматически.'
        )


## 3. Последовательность и признаки всей истории

Подготовим два вида признаков для нейросети:

- **Последовательные** - коды событий и продуктов, типы действий и временные характеристики последних 512 событий для GRU
- **Агрегированные** - общая активность клиента и статистики взаимодействий с каждым продуктом за всю доступную историю для MLP

Также учитываем давность действий, недавнюю активность и повторяемость интереса. Подготовленные массивы сохраняем в кэш, чтобы не пересчитывать признаки при повторных запусках

In [ ]:
# Признаки всей истории клиента (30)
GLOBAL_NAMES = [
    # Размер истории и разнообразие продуктов
    'log_length', 'log_duration_hours', 'log_unique_products', 'known_product_rate',

    # Доли различных типов действий
    'view_rate', 'click_rate', 'target_rate', 'other_rate', 'boundary_rate',

    # Сессии и распределение активности по времени
    'log_sessions', 'last_session_fraction', 'log_active_days', 'log_events_per_day',
    'log_gap_mean', 'log_gap_std', 'log_gap_median', 'log_gap_max', 'missing_time_rate',

    # Повторяемость и концентрация интереса к продуктам
    'click_repeat', 'target_repeat',
    'click_entropy', 'target_entropy',
    'click_concentration', 'target_concentration',

    # Активность в последних 20 и 100 событиях
    'last20_click_rate', 'last20_target_rate',
    'last100_click_rate', 'last100_target_rate',

    # Наличие целевых действий и доля непросмотровых событий
    'has_target', 'nonview_rate'
]


# Признаки каждого из 32 целевых продуктов (27 на продукт)
ACTION_NAMES = [
    'log_count',        # количество действий
    'share',            # доля среди действий данного типа
    'last20_share',     # доля в последних 20 событиях
    'last100_share',    # доля в последних 100 событиях
    'decayed_share',    # доля с уменьшением веса старых действий
    'log_age_hours',    # время с последнего действия
    'time_missing',     # отсутствует временная метка
    'seen',             # было ли действие с этим продуктом
    'log_sessions',     # число сессий с действиями
    'growth'            # изменение интереса относительно всей истории
]

# первые 20 признаков: по 10 отдельно для click и target
PRODUCT_NAMES = [
    f'{kind}_{name}'
    for kind in ['click', 'target']
    for name in ACTION_NAMES
] + [
    # Общая информация о взаимодействии с продуктом
    'log_age_any_hours',
    'time_missing_any',
    'last_position',
    'seen_any',

    # Совпадение с последними продуктами в истории
    'last_product',
    'last_click_product',
    'last_target_product'
]

def history_arrays(times, events, products):
    n, C = len(times), len(classes)
    kinds = event_types[events]
    click, target = kinds == 2, event_targets[events] > 0
    known, valid_time = np.isfinite(products), np.isfinite(times)

    # проверяем порядок событий: переставлять их нельзя, так как GRU учитывает хронологию
    assert np.all(np.diff(times[valid_time]) >= 0), 'Нарушен хронологический порядок'

    end = times[valid_time][-1] if valid_time.any() else np.nan
    gap = np.full(n, np.nan)
    known_time_idx = np.flatnonzero(valid_time)
    gap[known_time_idx[1:]] = np.diff(times[valid_time])
    age = np.where(valid_time, np.maximum(end - times, 0), 0)

    # разделяем историю на сессии по перерывам более 30 минут
    sessions = np.cumsum(np.nan_to_num(gap, nan=0) > 1800)
    last_session = sessions == sessions[-1] if n else np.zeros(0, bool)
    duration = np.ptp(times[valid_time]) / 3600 if valid_time.any() else 0
    days = len(np.unique(np.floor(times[valid_time] / 86400)))

    good_gaps = gap[np.isfinite(gap)] / 60
    gap_stats = [np.mean(good_gaps), np.std(good_gaps), np.median(good_gaps), np.max(good_gaps)] if len(good_gaps) else [0]*4

    # насколько действия распределены между продуктами
    concentration = []
    for mask in [click, target]:
        _, counts = np.unique(products[mask & known], return_counts=True)
        shares = counts / max(counts.sum(), 1)
        concentration.append((1 - len(counts)/max(counts.sum(), 1) if len(counts) else 0,
                              -(shares*np.log(shares)).sum(), shares.max() if len(shares) else 0))

    # общие признаки активности клиента за всю доступную историю
    g = [np.log1p(n), np.log1p(duration), np.log1p(len(np.unique(products[known]))), known.mean() if n else 0]
    g += [mask.mean() if n else 0 for mask in [kinds==1, click, target, kinds==3, kinds==4]]
    g += [np.log1p(sessions[-1]+1) if n else 0, last_session.mean() if n else 0,
          np.log1p(days), np.log1p(n/max(days,1)), *np.log1p(gap_stats), 1-valid_time.mean() if n else 0]
    g += [concentration[0][0], concentration[1][0], concentration[0][1], concentration[1][1], concentration[0][2], concentration[1][2]]
    g += [mask[-w:].mean() if n else 0 for w in [20,100] for mask in [click,target]]
    g += [float(target.any()), (kinds!=1).mean() if n else 0]
    assert len(g) == len(GLOBAL_NAMES)

    # сопоставляем продукты из истории с 32 возможными целевыми классами
    candidate = np.searchsorted(classes, np.nan_to_num(products, nan=-1))
    match = (candidate < C) & known
    match &= classes[np.minimum(candidate, C-1)] == products
    candidate = np.where(match, candidate, -1)

    def counts(mask, weights=None):
        ok = mask & match
        return np.bincount(candidate[ok], weights=None if weights is None else weights[ok], minlength=C).astype(np.float32)

    # для каждого продукта считаем частоты, давность, сессии и изменение интереса
    panel = []
    for mask in [click, target]:
        total = counts(mask)
        share = total / max(mask.sum(), 1)
        recent = []

        for w in [20,100]:
            tail = np.arange(n) >= max(0,n-w)
            recent.append(counts(mask & tail) / max((mask & tail).sum(), 1))

        # более старые действия получают меньший вес
        weight = np.where(valid_time, np.exp2(-age/86400), 0)
        decayed = counts(mask, weight) / max(weight[mask].sum(), 1e-8)

        ages, missing, seen, ns = np.zeros(C), np.ones(C), total>0, np.zeros(C)
        for j in np.flatnonzero(seen):
            idx = np.flatnonzero(mask & (candidate==j))
            last = idx[-1]
            if valid_time[last]: ages[j], missing[j] = np.log1p(age[last]/3600), 0
            ns[j] = np.log1p(len(np.unique(sessions[idx])))

        panel += [np.log1p(total), share, *recent, decayed, ages, missing, seen, ns, recent[1]-share]

    any_age, missing, position, seen = np.zeros(C), np.ones(C), np.zeros(C), np.zeros(C)
    for j in np.unique(candidate[match]):
        idx = np.flatnonzero(candidate==j)[-1]
        seen[j], position[j] = 1, (n-1-idx)/max(n,1)
        if valid_time[idx]: any_age[j], missing[j] = np.log1p(age[idx]/3600), 0

    # отмечаем последние продукты в истории и в действиях разных типов
    last_products = []
    for mask in [known, click & known, target & known]:
        idx = np.flatnonzero(mask)
        last_products.append((classes == products[idx[-1]]).astype(np.float32) if len(idx) else np.zeros(C))

    panel = np.column_stack(panel + [any_age, missing, position, seen, *last_products]).astype(np.float32)
    assert panel.shape == (C, len(PRODUCT_NAMES))

    # профиль непросмотровых действий клиента
    activity = np.bincount(events[kinds!=1], minlength=len(event_types)).astype(np.uint16)

    # 6 числовых признаков для каждого события последовательности
    numerics = np.column_stack([np.log1p(np.nan_to_num(gap, nan=0))/16,
        np.log1p(age)/16, valid_time, event_targets[events],
        np.where(valid_time, np.sin(2*np.pi*np.nan_to_num(times)/86400), 0),
        np.where(valid_time, np.cos(2*np.pi*np.nan_to_num(times)/86400), 0)]).astype(np.float16)

    return np.asarray(g,np.float32), panel, activity, numerics, end


In [ ]:
cache_signature = hashlib.sha256(json.dumps(dict(version=PREPROCESS_VERSION,
    sources=source_hashes, length=CFG['max_len'], smoke=SMOKE_TEST), sort_keys=True).encode()).hexdigest()

CACHE = WORK / 'arrays'

# ищем готовый совместимый кэш, чтобы не пересчитывать признаки
cache_candidates = [CACHE] + [p.parent for p in INPUT_ROOT.rglob('gru_cache.json')]
cache_info = None

for candidate in cache_candidates:
    manifest = candidate / 'gru_cache.json'
    if manifest.exists():
        info = json.loads(manifest.read_text())
        if info['signature'] == cache_signature and all((candidate / f'{s}_{k}.npy').exists()
            for s in ['train','test'] for k in ['tokens','time','length','global','product','activity','end']):
            CACHE, cache_info = candidate, info
            product_map = {int(k):v for k,v in info['product_map'].items()}
            break


def prepare_file(path, size, prefix):
    # создаём массивы на диске через memmap
    shapes = dict(tokens=((size,CFG['max_len'],3),'uint16'), time=((size,CFG['max_len'],6),'float16'),
        length=((size,),'uint16'), global_=((size,len(GLOBAL_NAMES)),'float32'),
        product=((size,len(classes),len(PRODUCT_NAMES)),'float32'),
        activity=((size,len(event_types)),'uint16'), end=((size,),'float64'))

    arrays = {key.rstrip('_'): np.lib.format.open_memmap(CACHE / f'{prefix}_{key.rstrip("_")}.npy',
                mode='w+',dtype=dtype,shape=shape) for key,(shape,dtype) in shapes.items()}

    offset = 0

    # читаем длинные истории небольшими пакетами из Parquet
    for batch in tqdm(pq.ParquetFile(path).iter_batches(batch_size=CFG['arrow_batch'], columns=['events'], use_threads=False),
                      total=math.ceil(size/CFG['arrow_batch']), desc=prefix):
        lists = batch.column('events'); values = lists.values
        offsets = lists.offsets.to_numpy()

        # переводим миллисекунды в секунды
        ts = values.field('timestamp').to_numpy(zero_copy_only=False).astype(np.float64) * 0.001
        raw_event = values.field('encoded_event').to_numpy(zero_copy_only=False)
        raw_product = values.field('encoded_product').to_numpy(zero_copy_only=False).astype(np.float64)

        unique_events, inverse = np.unique(raw_event, return_inverse=True)
        event_ids = np.asarray([event_map.get(int(v),1) if np.isfinite(v) else 1 for v in unique_events])[inverse]

        # добавляем встреченные продукты в словарь embedding-индексов
        for value in np.unique(raw_product[np.isfinite(raw_product)]):
            product_map.setdefault(int(value), len(product_map)+3)

        product_ids = np.ones(len(raw_product),dtype=np.uint16)
        for value in np.unique(raw_product[np.isfinite(raw_product)]):
            product_ids[raw_product==value] = product_map[int(value)]

        assert max(product_map.values()) < 65536

        for i in range(min(len(batch), size-offset)):
            start, stop = offsets[i:i+2]
            t,e,p = ts[start:stop],event_ids[start:stop],raw_product[start:stop]
            g, panel, activity, numbers, end = history_arrays(t,e,p)

            row, length = offset+i, min(len(e),CFG['max_len'])
            arrays['global'][row],arrays['product'][row],arrays['activity'][row],arrays['end'][row] = g,panel,activity,end

            arrays['tokens'][row] = 0; arrays['time'][row] = 0
            arrays['length'][row] = max(length,1)

            # GRU получает последние max_len событий, всю историю
            if length:
                arrays['tokens'][row,:length] = np.column_stack([e[-length:],product_ids[start:stop][-length:],event_types[e[-length:]]])
                arrays['time'][row,:length] = numbers[-length:]
            else:
                # Для пустой истории создаём одно нейтральное событие
                arrays['tokens'][row,0] = [1,1,5]

        offset += min(len(batch),size-offset)
        if offset == size: break

    assert offset == size
    for array in arrays.values(): array.flush()
    del arrays


# при отсутствии совместимого кэша рассчитываем признаки заново
if cache_info is None:
    CACHE.mkdir(parents=True,exist_ok=True)
    prepare_file(DATA_DIR/'train.parquet',len(train_meta),'train')
    prepare_file(DATA_DIR/'test.parquet',len(test_meta),'test')

    cache_info = dict(signature=cache_signature, product_map=product_map, classes=classes.tolist(),
                      global_names=GLOBAL_NAMES, product_names=PRODUCT_NAMES)

    (CACHE/'gru_cache.json').write_text(json.dumps(cache_info,ensure_ascii=False,indent=2))

# открываем массивы без полной загрузки в оперативную память
arrays = {split:{key:np.load(CACHE/f'{split}_{key}.npy',mmap_mode='r')
                 for key in ['tokens','time','length','global','product','activity','end']} for split in ['train','test']}

assert cache_info['classes'] == classes.tolist()
assert cache_info['global_names'] == GLOBAL_NAMES and cache_info['product_names'] == PRODUCT_NAMES

print('Продуктовых кодов:',len(product_map), '| кеш, ГБ:',round(sum(p.stat().st_size for p in CACHE.glob('*.npy'))/1024**3,2))


## 4. Подготовка входов модели

Выберем 16 наиболее частых непросмотровых действий и добавим их к общим признакам клиента

Нормализуем числовые признаки, рассчитывая средние значения и стандартные отклонения только на обучающей части (fit), чтобы избежать утечки данных.

Затем создадим `Dataset` и `DataLoader` для формирования батчей с последовательностями событий и агрегированными признаками.

In [ ]:
# выбираем самые частые непросмотровые действия по обучающей выборке
profile_sum = np.zeros(len(event_types), np.float64)
for start in range(0, len(train_idx), 2048):
    block = np.asarray(arrays['train']['activity'][train_idx[start:start+2048]], np.float32)
    profile_sum += (block / np.maximum(block.sum(axis=1, keepdims=True), 1)).sum(axis=0)

activity_cols = np.argsort(-profile_sum, kind='stable')[:CFG['activity_top']]
activity_cols = activity_cols[profile_sum[activity_cols] > 0]


def global_inputs(block):
    activity = np.asarray(block['activity'], np.float32)
    count = activity.sum(axis=1, keepdims=True)

    selected = activity[:, activity_cols] / np.maximum(count, 1)
    return np.column_stack([
        block['global'],
        selected,
        np.where(count[:, 0] > 0, 1 - selected.sum(axis=1), 0),
        count[:, 0] > 0
    ]).astype(np.float32)


# собираем общие признаки (для train и test)
global_values = {}
for s in ['train', 'test']:
    parts = []
    for start in range(0, len(arrays[s]['global']), 2048):
        parts.append(global_inputs({
            key: arrays[s][key][start:start+2048]
            for key in ['global', 'activity']
        }))
    global_values[s] = np.concatenate(parts)
    del parts


# среднее и стандартное отклонение считаем только на fit
g_mean = global_values['train'][train_idx].mean(axis=0)
g_std = np.maximum(global_values['train'][train_idx].std(axis=0), 0.05)


# нормализация признаков 
p_sum = np.zeros(len(PRODUCT_NAMES))
p_sq = np.zeros(len(PRODUCT_NAMES))
n_product_rows = 0

for start in range(0, len(train_idx), 1024):
    block = np.asarray(
        arrays['train']['product'][train_idx[start:start+1024]],
        np.float64
    ).reshape(-1, len(PRODUCT_NAMES))

    p_sum += block.sum(axis=0)
    p_sq += (block * block).sum(axis=0)
    n_product_rows += len(block)

p_mean = (p_sum / n_product_rows).astype(np.float32)
p_std = np.maximum(
    np.sqrt(np.maximum(p_sq / n_product_rows - p_mean.astype(float)**2, 0)),
    0.05
).astype(np.float32)


# приводим общие признаки к единому масштабу и ограничиваем выбросы
for s in global_values:
    global_values[s] = np.clip(
        (global_values[s] - g_mean) / g_std, -10, 10
    ).astype(np.float32)

class_tokens = np.asarray([product_map[int(c)] for c in classes])

print('Общих признаков:', global_values['train'].shape[1],
      '| на продукт:', len(PRODUCT_NAMES))


In [ ]:
class HistoryDataset(Dataset):
    def __init__(self, split, rows):
        self.split, self.rows = split, np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        data = arrays[self.split]
        length = int(data['length'][row])

        # нормализуем признаки 
        product = np.clip(
            (np.asarray(data['product'][row]) - p_mean) / p_std,
            -10, 10
        ).astype(np.float32)

        # Возвращаем последовательность, агрегаты, ответ и индекс клиента
        return (
            np.array(data['tokens'][row, :length], dtype=np.int64),
            np.array(data['time'][row, :length], dtype=np.float32),
            global_values[self.split][row],
            product,
            int(y[row]) if self.split == 'train' else -1,
            int(row)
        )


def collate_histories(samples):
    lengths = torch.tensor([len(s[0]) for s in samples], dtype=torch.long)
    tokens = torch.zeros(len(samples), int(lengths.max()), 3, dtype=torch.long)
    time = torch.zeros(len(samples), int(lengths.max()), 6)

    for i, s in enumerate(samples):
        tokens[i, :lengths[i]] = torch.from_numpy(s[0])
        time[i, :lengths[i]] = torch.from_numpy(s[1])

    # собираем все входы нейросети в один батч
    return dict(
        tokens=tokens,
        time=time,
        lengths=lengths,
        global_=torch.from_numpy(np.stack([s[2] for s in samples])),
        product=torch.from_numpy(np.stack([s[3] for s in samples])),
        label=torch.tensor([s[4] for s in samples]),
        row=torch.tensor([s[5] for s in samples])
    )


def loader(split, rows, shuffle=False, seed=42):
    # DataLoader формирует батчи и при необходимости перемешивает примеры
    return DataLoader(
        HistoryDataset(split, rows),
        batch_size=CFG['batch_size'],
        shuffle=shuffle,
        collate_fn=collate_histories,
        num_workers=0,
        pin_memory=AMP,
        generator=torch.Generator().manual_seed(seed)
    )


## 5. Архитектура GRU

Модель объединяет последовательные и агрегированные признаки клиента

Последние 512 событий преобразуются с помощью **embeddings** и обрабатываются двухслойной **GRU** со скрытым состоянием размерности 128. Для представления последовательности используются последнее скрытое состояние и **attention pooling**, который выделяет наиболее информативные события

Параллельно две **MLP** обрабатывают признаки взаимодействий с отдельными продуктами и общие характеристики всей истории клиента. Полученные представления объединяются, после чего модель вычисляет оценки для 32 целевых продуктов

In [ ]:
class ProductGRU(nn.Module):
    def __init__(self):
        super().__init__()

        # embedding-слои превращают категориальные индексы в обучаемые векторы
        self.event_emb = nn.Embedding(len(event_types), 48, padding_idx=0)
        self.product_emb = nn.Embedding(max(product_map.values()) + 1, 32, padding_idx=0)
        self.type_emb = nn.Embedding(6, 8, padding_idx=0)

        self.input = nn.Sequential(
            nn.Linear(94, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # обрабатываем события в хронологическом порядке
        self.gru = nn.GRU(
            96, CFG['hidden'],
            num_layers=CFG['layers'],
            batch_first=True,
            dropout=0.15 if CFG['layers'] > 1 else 0
        )

        self.attention = nn.Linear(CFG['hidden'], 1)

        # MLP для 27 агрегированных признаков каждого продукта
        self.product_net = nn.Sequential(
            nn.Linear(len(PRODUCT_NAMES), 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(CFG['dropout']),
            nn.Linear(64, 32)
        )

        # MLP для общих признаков истории и усреднённых признаков продуктов
        self.global_net = nn.Sequential(
            nn.Linear(global_values['train'].shape[1] + 32, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(CFG['dropout'])
        )

        # объединяем последнее состояние GRU, attention pooling и глобальные признаки
        self.fusion = nn.Sequential(
            nn.Linear(CFG['hidden'] * 2 + 96, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(CFG['dropout'])
        )

        self.query = nn.Linear(128, 32)
        self.head = nn.Linear(128, len(classes))

        # индексы 32 целевых продуктов хранятся вместе с моделью,
        self.register_buffer(
            'class_tokens',
            torch.tensor(class_tokens, dtype=torch.long)
        )

    def forward(self, batch):
        token = batch['tokens']
        length = batch['lengths']

        # три embedding-вектора + шесть временных признаков
        x = torch.cat([
            self.event_emb(token[:, :, 0]),
            self.product_emb(token[:, :, 1]),
            self.type_emb(token[:, :, 2]),
            batch['time']
        ], dim=-1)
        x = self.input(x)

        # упаковываем последовательности, чтобы GRU
        # не обрабатывала добавленные при padding пустые позиции
        packed = pack_padded_sequence(
            x, length.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        out, hidden = self.gru(packed)

        # возвращаем скрытые состояния к форме обычного батча
        out, _ = pad_packed_sequence(
            out,
            batch_first=True,
            total_length=token.shape[1]
        )

        # attention pooling
        mask = (
            torch.arange(token.shape[1], device=token.device)[None, :]
            < length.to(token.device)[:, None]
        )
        weight = torch.softmax(
            self.attention(out).squeeze(-1).float().masked_fill(~mask, -1e9),
            dim=1
        )

        # взвешенная сумма состояний GRU - представление последовательности
        pooled = (out * weight.unsqueeze(-1)).sum(dim=1)

        # обрабатываем признаки через MLP
        product = self.product_net(batch['product'])

        # Усредняем представления продуктов и объединяем
        global_ = self.global_net(
            torch.cat([batch['global_'], product.mean(dim=1)], dim=-1)
        )

        # последнее состояние GRU + attention pooling + глобальные признаки
        context = self.fusion(
            torch.cat([hidden[-1], pooled, global_], dim=-1)
        )

        candidates = product + self.product_emb(self.class_tokens)[None, :, :]

        return (
            self.head(context)
            + (candidates * self.query(context)[:, None, :]).sum(-1)
              / math.sqrt(32)
        )


# создаём модель и выводим количество обучаемых параметров
seed_everything(CFG['seed'])
model = ProductGRU().to(DEVICE)

print(f'Параметров модели: {sum(p.numel() for p in model.parameters()):,}')


## 6. Обучение GRU

Обучим модель с помощью **AdamW** и функции потерь **CrossEntropyLoss**

После каждой эпохи оцениваем Accuracy на выборке ES. Если качество не улучшается три эпохи подряд, обучение останавливается (**early stopping**)

Сохраняем два checkpoint: `best.pt` с лучшими весами модели и `last.pt` для продолжения прерванного обучения. Отложенная выборка outer на этом этапе не используется

In [ ]:
def to_device(batch):
    return {
        key: (value if key in ['lengths', 'row'] else value.to(DEVICE, non_blocking=AMP))
        for key, value in batch.items()
    }


# получаем вероятности классов без вычисления градиентов
@torch.no_grad()
def predict(rows, split='train'):
    model.eval()
    result = []
    actual_rows = []

    for batch in loader(split, rows):
        actual_rows.append(batch['row'].numpy())

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            logits = model(to_device(batch))

        prob = logits.float().softmax(-1).cpu().numpy()
        assert np.isfinite(prob).all()
        result.append(prob)

    assert np.array_equal(np.concatenate(actual_rows), rows)
    return np.concatenate(result)


# сохраняем checkpoint через временный файл, чтобы избежать повреждения при прерывании
def atomic_save(value, path):
    temporary = path.with_suffix('.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)


# сохраняем параметры предобработки и разбиения для воспроизводимости
preprocessing = dict(
    g_mean=g_mean, g_std=g_std,
    p_mean=p_mean, p_std=p_std,
    activity_cols=activity_cols,
    event_map=event_map, product_map=product_map,
    class_tokens=class_tokens, classes=classes,
    global_names=GLOBAL_NAMES, product_names=PRODUCT_NAMES,
    source_hashes=source_hashes,
    train_idx=train_idx, valid_idx=valid_idx,
    es_idx=es_idx, outer_idx=outer_idx,
    versions=dict(
        torch=str(torch.__version__),
        numpy=np.__version__,
        pandas=pd.__version__
    )
)

run_signature = hashlib.sha256(
    json.dumps(
        dict(
            cache=cache_signature,
            config=CFG,
            train_idx=train_idx.tolist(),
            es_idx=es_idx.tolist(),
            activity_cols=activity_cols.tolist()
        ),
        sort_keys=True
    ).encode()
).hexdigest()


# AdamW, уменьшение learning rate и CrossEntropyLoss
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG['epochs'],
    eta_min=CFG['lr'] * 0.05
)

scaler = torch.amp.GradScaler('cuda', enabled=AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

start_epoch, best_accuracy, bad_epochs, logs = 0, -1.0, 0, []
last_path, best_path = WORK / 'last.pt', WORK / 'best.pt'

resume_candidates = [last_path] + sorted(INPUT_ROOT.rglob('last.pt'))

# загружаем только собственные checkpoint, сохранённые этим ноутбуком
for path in resume_candidates:
    if not path.exists():
        continue

    checkpoint = torch.load(path, map_location='cpu', weights_only=False)

    if checkpoint.get('signature') != run_signature:
        continue

    source_best = path.with_name('best.pt')
    if not source_best.exists():
        raise FileNotFoundError('Для продолжения нужны last.pt и best.pt')

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])

    start_epoch = checkpoint['epoch'] + 1
    best_accuracy = checkpoint['best_accuracy']
    bad_epochs = checkpoint['bad_epochs']
    logs = checkpoint['logs']

    if source_best.resolve() != best_path.resolve():
        shutil.copyfile(source_best, best_path)

    print('Продолжаем с эпохи', start_epoch + 1)
    del checkpoint
    break


if DEVICE.type != 'cuda' and not SMOKE_TEST:
    raise RuntimeError('Полное обучение на CPU отключено. Включите GPU в настройках Kaggle.')


# основной цикл обучения
for epoch in range(start_epoch, CFG['epochs']):
    if bad_epochs >= CFG['patience']:
        break

    seed_everything(CFG['seed'] + epoch)
    model.train()

    total_loss = 0.
    total_correct = 0
    total = 0
    started = perf_counter()

    for batch in tqdm(
        loader('train', train_idx, shuffle=True, seed=CFG['seed'] + epoch),
        desc=f'Эпоха {epoch+1}'
    ):
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        # Прямой проход и вычисление функции потерь
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP):
            logits = model(batch)
            loss = criterion(logits, batch['label'])

        if not torch.isfinite(loss):
            raise FloatingPointError('Loss не конечен; checkpoint предыдущей эпохи сохранён')

        # обратное распространение ошибки и обновление весов
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        norm = nn.utils.clip_grad_norm_(
            model.parameters(),
            CFG['grad_clip'],
            error_if_nonfinite=not AMP
        )

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(batch['label'])
        total += len(batch['label'])
        total_correct += (logits.argmax(1) == batch['label']).sum().item()

    # оцениваем качество на ES после каждой эпохи
    es_probability = predict(es_idx)
    es_accuracy = accuracy_score(y[es_idx], es_probability.argmax(1))

    es_loss = -np.log(
        np.maximum(
            es_probability[np.arange(len(es_idx)), y[es_idx]],
            1e-12
        )
    ).mean()

    logs.append(dict(
        epoch=epoch + 1,
        train_loss=total_loss / total,
        train_accuracy=total_correct / total,
        es_loss=float(es_loss),
        es_accuracy=float(es_accuracy),
        seconds=perf_counter() - started
    ))

    scheduler.step()

    # сохраняем лучшие веса и сбрасываем счётчик эпох без улучшения
    if es_accuracy > best_accuracy:
        best_accuracy, bad_epochs = es_accuracy, 0

        atomic_save(
            dict(
                model=model.state_dict(),
                epoch=epoch,
                best_accuracy=best_accuracy,
                signature=run_signature,
                config=CFG,
                preprocessing=preprocessing
            ),
            best_path
        )
    else:
        bad_epochs += 1

    # last.pt содержит также оптимизатор и scheduler для продолжения обучения
    atomic_save(
        dict(
            model=model.state_dict(),
            optimizer=optimizer.state_dict(),
            scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),
            epoch=epoch,
            best_accuracy=best_accuracy,
            bad_epochs=bad_epochs,
            logs=logs,
            signature=run_signature,
            config=CFG,
            preprocessing=preprocessing
        ),
        last_path
    )

    print(pd.Series(logs[-1]).round(5).to_string())

display(pd.DataFrame(logs).round(5))


## 7. Итоговая оценка

Загрузим веса модели с лучшей Accuracy на ES и оценим её качество на отложенной выборке **outer**, которая не использовалась при обучении и выборе эпохи

Дополнительно проверим качество на длинных и более поздних историях клиентов, чтобы оценить устойчивость модели к различиям в данных

In [ ]:
checkpoint = torch.load(best_path,map_location='cpu',weights_only=False)
assert checkpoint['signature'] == run_signature
model.load_state_dict(checkpoint['model']); model.to(DEVICE)
valid_probability = predict(valid_idx)
valid_position = {int(row):i for i,row in enumerate(valid_idx)}
outer_pos = np.asarray([valid_position[int(row)] for row in outer_idx])
lengths = np.expm1(arrays['train']['global'][outer_idx,0])
fit_end = arrays['train']['end'][train_idx]
late_threshold = np.nanquantile(fit_end,0.8)
slices = {'Outer':np.ones(len(outer_idx),bool),'Истории ≥1000 событий':lengths>=999.5,
          'Поздние истории':arrays['train']['end'][outer_idx]>=late_threshold}
results = []
for name,mask in slices.items():
    if mask.any(): results.append(dict(Выборка=name,Историй=int(mask.sum()),
        Accuracy=accuracy_score(y[outer_idx[mask]],valid_probability[outer_pos[mask]].argmax(1))))
most_frequent = np.bincount(y[train_idx],minlength=len(classes)).argmax()
print('Самый частый продукт, outer Accuracy:',round(float((y[outer_idx]==most_frequent).mean()),5))
display(pd.DataFrame(results))
print('Лучшая эпоха:',checkpoint['epoch']+1)
del checkpoint

## 8. Предсказания и сохранение результатов

Получим предсказания GRU для тестовой выборки и выберем продукт с наибольшей вероятностью для каждого клиента

Преобразуем индексы классов обратно в исходные коды продуктов и сохраним результаты в CSV в формате Kaggle, сохранив порядок `example_id` из образца отправки

При локальном запуске CSV сохраняются в `task3/submits/`, на Kaggle — непосредственно в `/kaggle/working/`. Кеш и checkpoints остаются в каталоге `WORK`; существующие CSV не перезаписываются.


In [ ]:
def save_submission(probability,name):
    assert probability.shape==(len(test_meta),len(classes))
    assert np.isfinite(probability).all() and (probability>=0).all()
    assert np.allclose(probability.sum(axis=1),1,atol=1e-5)
    sample = pd.read_csv(DATA_DIR/'sample_submission.csv',dtype={'example_id':str})
    assert list(sample.columns)==['example_id','encoded_product'] and sample.example_id.is_unique
    predicted = pd.DataFrame({'example_id':test_meta.example_id.astype(str),
                              'encoded_product':classes[probability.argmax(axis=1)]})
    assert predicted.example_id.is_unique and set(sample.example_id)==set(predicted.example_id)
    submission = sample[['example_id']].merge(predicted,on='example_id',how='left',validate='one_to_one',sort=False)
    assert submission.example_id.equals(sample.example_id)
    assert len(submission)==len(test_meta) and not submission.isna().any().any()
    assert submission.encoded_product.isin(classes).all()
    path = SUBMISSION_DIR/name
    suffix=1
    while path.exists():
        path=SUBMISSION_DIR/f'{Path(name).stem}_{suffix}.csv'; suffix+=1
    submission.to_csv(path,index=False,mode='x')
    print('Сохранён:',path,'| строк:',len(submission))
    return path

if not SMOKE_TEST:
    test_probability = predict(np.arange(len(test_meta)),split='test')
    np.savez_compressed(WORK/'gru_probabilities.npz',classes=classes,
        valid_ids=train_meta.iloc[valid_idx].example_id.to_numpy(dtype=str),test_ids=test_meta.example_id.to_numpy(dtype=str),
        valid_proba=valid_probability,test_proba=test_probability,
        train_idx=train_idx,valid_idx=valid_idx,es_idx=es_idx,outer_idx=outer_idx,signature=run_signature)
    gru_submission = save_submission(test_probability,'submission_gru.csv')
else:
    print('Smoke test: CSV и test-предсказания для соревнования не создаём.')

## 9. Ансамбль GRU и LightGBM

В оригинальном Kaggle-ноутбуке `notebook01adca8a4e.ipynb` этот ансамбль создал `submission_gru_lgbm_blend.csv` с **Public Score 0.40216**. LightGBM-компонент — смесь двух Ranker на **0.37191**: 74 и 92 признака пары, веса 50/50. Это не одиночный `interest_v2` на 0.37067.

При сравнении эталона с текущим ноутбуком вычисления признаков, архитектура, Dataset, нормализация и шаги обучения совпали. Изменены пути, способ проверки разбиения и временной ограничитель. В эталонном запуске ограничитель не сработал: полностью завершены 10 эпох, лучшей стала эпоха 7. Разделы 1–8 здесь не изменяем; совпадение численного результата после нового GPU-обучения требует проверки.

В режиме `train` обучаем оба Ranker без старых весов. Совместимый кэш необязателен: если его нет, рассчитываем агрегаты из Parquet пакетами по 32 истории. `saved` предназначен для проверки исторических весов на их исходном кэше, `bridge` — ускоренный путь оригинального Kaggle-ноутбука.

По умолчанию раздел включён и обучает LightGBM заново. Выполняйте ноутбук сверху вниз; совместимый кэш используется при наличии, иначе признаки рассчитываются из Parquet. Для самостоятельной GRU можно отключить `RUN_LGBM_ENSEMBLE`. Дополнительные Python-скрипты не нужны.


In [ ]:
RUN_LGBM_ENSEMBLE = True
RANKER_SOURCE = 'train'  # 'saved' — исторические веса; 'bridge' — готовые вероятности
RANKER_CACHE_FILE = None  # при необходимости Path к aggregates.joblib
RANKER_BUNDLE_FILE = None  # selected_ensemble.joblib, только для saved
USE_RANKER_FEATURE_CACHE = True  # False — пересчитать из Parquet
SAVE_RANKER_FEATURE_CACHE = True


In [ ]:
if RUN_LGBM_ENSEMBLE and RANKER_SOURCE != 'bridge':
    import gc, mmap, re, time
    import joblib
    import lightgbm as lgb
    from scipy.sparse import csr_matrix
    from scipy.optimize import minimize_scalar
    from scipy.special import logsumexp

    BATCH = 128
    CANONICAL_SEED = 42
    BASE_GLOBALS = ["history_length", "click_rate", "view_rate", "n_unique_products",
                    "known_product_rate", "history_duration_hours", "interest_n_sessions",
                    "interest_last_session_length"]

    BASE_SUFFIXES = [
        "click", "target_event_count", "hours_since_last", "hours_since_last_target_event",
        "last_time_missing", "last_target_event_time_missing", "last_position",
        "last20_click", "last100_click", "last20_target_event", "last100_target_event",
        "interest_click_view_ratio", "interest_target_view_ratio", "interest_click_share", "interest_target_share",
        "interest_last20_click_view_ratio", "interest_last100_click_view_ratio",
        "interest_last20_target_view_ratio", "interest_last100_target_view_ratio",
        "interest_recent_click_fraction", "interest_recent_target_fraction",
        "interest_click_growth", "interest_target_growth",
        "interest_decayed_click_share", "interest_decayed_target_share",
        "interest_n_click_sessions", "interest_n_target_sessions",
        "interest_last_session_click_share", "interest_last_session_target_share",
    ]

    CATEGORIES = ["last_product", "last_click_product", "last_view_product",
                  "last_target_event_product", "most_frequent_product"]

    TIME_SUFFIXES = [
        "v2_hours_since_click", "v2_click_time_missing", "v2_hours_since_previous_target",
        "v2_previous_target_time_missing", "v2_target_gap_mean_hours", "v2_target_gap_count",
        "v2_target_1d_share", "v2_target_3d_share", "v2_target_7d_share",
        "v2_click_1d_share", "v2_click_3d_share", "v2_click_7d_share",
        "v2_target_1d", "v2_target_3d", "v2_target_7d",
        "v2_steps_since_click", "v2_steps_since_target",
    ]

    DERIVED_TIME = ["target_recency_fraction", "target_recency_cycles_log",
                    "target_fraction_of_clicks", "target_day_fraction_of_week"]

    BASE_PARAMS = dict(
        objective="lambdarank", metric="ndcg", n_estimators=500, learning_rate=0.05,
        num_leaves=15, max_depth=5, min_child_samples=100, reg_lambda=5,
        max_bin=63, force_col_wise=True, histogram_pool_size=128, zero_as_missing=False,
        label_gain=[0, 1], importance_type="gain", n_jobs=2, random_state=42, verbosity=-1,
    )

    def drop_pages(array):
        # Только read-only mmap текущего процесса: файлы и пользовательское ядро не меняются.
        seen = set()
        while array is not None and id(array) not in seen:
            seen.add(id(array))
            mapping = getattr(array, "_mmap", None)
            if mapping is not None and hasattr(mapping, "madvise"):
                try:
                    mapping.madvise(mmap.MADV_DONTNEED)
                except (OSError, ValueError):
                    pass
                return
            array = getattr(array, "base", None)

    class RankerData:
        def __init__(self, store):
            self.train, self.test = train_meta, test_meta
            self.store = store
            assert self.store['train_ids'] == self.train.example_id.tolist()
            assert self.store['test_ids'] == self.test.example_id.tolist()
            assert self.store['signature']['version'] == 'v4_temporal_interest_20261110'
            if self.store['signature']['code'] != '38fa868c9b8d9f2360468ed2ffd4c7adceb4c96b3882088eb3acd78182511284':
                assert self.store['signature'] == ranker_generator_signature
            assert self.store['signature']['dictionary'] == ranker_generator_signature['dictionary']
            assert joblib.hash(self.store['names']) == 'd7bd07e8a758a8419ad38182d6466134'
            self.x, self.xt = self.store['train'], self.store['test']
            self.names = self.store['names']
            assert self.x.shape == (len(self.train), len(self.names))
            assert self.xt.shape == (len(self.test), len(self.names))
            self.lookup = {name: i for i, name in enumerate(self.names)}
            self.codes = np.sort(self.train.target_product.unique())
            self.y = self.train.target_product.to_numpy()
            self.catalog = np.asarray(sorted({int(m.group(1)) for name in self.names
                if (m := re.match(r'p_(-?\d+)_', name))}), dtype=np.int32)
            self.catalog_index = {int(p): i for i, p in enumerate(self.catalog)}
            self.view = self.column('view_rate')

        def column(self, name, test=False):
                matrix = self.xt if test else self.x
                result = matrix[:, self.lookup[name]].toarray().ravel()
                self.release()
                return result

        def release(self):
                for matrix in (self.x, self.xt):
                    for array in (matrix.data, matrix.indices, matrix.indptr):
                        drop_pages(array)

        def training_presence(self, rows):
                presence = np.zeros(len(self.names), dtype=bool)
                for start in range(0, len(rows), 1024):
                    block = self.x[rows[start:start + 1024]]
                    presence[block.indices] = True
                    del block
                    self.release()
                return presence

    class Schema:
        def __init__(self, data, split, extended=False, encoding="onehot"):
            self.extended = extended
            self.encoding = encoding
            self.catalog = data.catalog
            self.catalog_index = data.catalog_index
            self.codes = data.codes
            self.id_allowed = set(map(int, split["pool"]))
            self.suffixes = BASE_SUFFIXES + (TIME_SUFFIXES if extended else [])
            presence = data.training_presence(split["fit"])
            dynamic = {"last20_click", "last100_click", "last20_target_event", "last100_target_event"}
            self.source = np.asarray([[data.lookup.get(f"p_{p}_{suffix}", -1) for suffix in self.suffixes]
                                      + [data.lookup.get(f"{name}={p}", -1) for name in CATEGORIES] for p in self.catalog])
            # Оригинальный DictVectorizer создавал динамические окна/категории только при наличии в fit.
            for j, p in enumerate(self.catalog):
                for k, suffix in enumerate(self.suffixes):
                    col = self.source[j, k]
                    if int(p) in self.id_allowed and suffix in dynamic and col >= 0 and not presence[col]:
                        self.source[j, k] = -1
                for k in range(len(self.suffixes), self.source.shape[1]):
                    col = self.source[j, k]
                    if int(p) in self.id_allowed and col >= 0 and not presence[col]:
                        self.source[j, k] = -1
            self.globals = np.asarray([data.lookup[name] for name in BASE_GLOBALS])
            identity_names = [f"candidate={p}" for p in self.codes] if encoding == "onehot" else ["candidate_category"]
            self.feature_names = BASE_GLOBALS + BASE_SUFFIXES + ["is_" + name for name in CATEGORIES] + identity_names
            if extended:
                self.feature_names += TIME_SUFFIXES + DERIVED_TIME
            assert len(self.feature_names) == 42 + len(identity_names) + (21 if extended else 0)

        @classmethod
        def from_state(cls, state):
            result = object.__new__(cls)
            result.__dict__.update(state)
            if not hasattr(result, "encoding"):
                result.encoding = "onehot"
            return result

        def panel(self, data, rows, test=False):
            block = (data.xt if test else data.x)[rows]
            globals_ = block[:, self.globals].toarray().astype(np.float32)
            flat = self.source.ravel()
            available = np.flatnonzero(flat >= 0)
            values = np.zeros((len(rows), len(flat)), dtype=np.float32)
            values[:, available] = block[:, flat[available]].toarray()
            values = values.reshape(len(rows), len(self.catalog), self.source.shape[1])
            return globals_, values

        def pairs(self, g, panel, candidates):
            sizes = np.asarray([len(row) for row in candidates], dtype=np.int32)
            q = np.repeat(np.arange(len(g)), sizes)
            indices = np.concatenate(candidates)
            p = panel[q, indices]
            ids = np.zeros((len(q), len(self.codes)), dtype=np.float32)
            for j, code in enumerate(self.codes):
                if int(code) in self.id_allowed:
                    ids[:, j] = self.catalog[indices] == code
            s = len(self.suffixes)
            if self.encoding == "categorical":
                candidate_codes = self.catalog[indices]
                known = np.isin(candidate_codes, list(self.id_allowed))
                identity = np.where(known, np.searchsorted(self.codes, candidate_codes) + 1, -1).astype(np.float32)[:, None]
            else:
                identity = ids
            parts = [g[q], p[:, :29], p[:, s:s + 5], identity]
            if self.extended:
                extra = p[:, 29:s]
                ix = {name: i for i, name in enumerate(TIME_SUFFIXES)}
                recency = p[:, BASE_SUFFIXES.index("hours_since_last_target_event")]
                has_time = p[:, BASE_SUFFIXES.index("last_target_event_time_missing")] == 0
                gap = extra[:, ix["v2_target_gap_mean_hours"]]
                gap_count = extra[:, ix["v2_target_gap_count"]]
                derived = np.column_stack([
                    np.where(has_time, recency / (g[q, 5] + 1), 0),
                    np.where(has_time & (gap_count > 0), np.log1p(recency / (gap + 1)), 0),
                    p[:, 1] / (p[:, 0] + 1),
                    extra[:, ix["v2_target_1d"]] / (extra[:, ix["v2_target_7d"]] + 1),
                ])
                parts += [extra, derived]
            matrix = np.column_stack(parts).astype(np.float32)
            assert matrix.shape[1] == len(self.feature_names) and np.isfinite(matrix).all()
            return csr_matrix(matrix), sizes

    class ActivitySchema(Schema):
        # Только activity_profile из сохранённого policy_experiments.py.
        def __init__(self, data, split):
            super().__init__(data, split)
            self.activity_names = []
            names = [name for name in data.names if re.fullmatch(r'v2_event_-?\d+_share', name)]
            columns = np.asarray([data.lookup[name] for name in names])
            total = np.zeros(len(columns), np.float64)
            for start in range(0, len(split['fit']), 512):
                rows = split['fit'][start:start + 512]
                total += (data.x[rows][:, columns].toarray()
                          / np.maximum(1 - data.view[rows, None], 1e-6)).sum(axis=0)
                data.release()
            selected = np.argsort(-total, kind='stable')[:16]
            self.activity_names = [names[i] for i in selected]
            self.activity_fit_mass = float(total[selected].sum() / max(total.sum(), 1e-6))
            self.globals = np.r_[self.globals, columns[selected]]
            self.feature_names += ['activity_' + name.removeprefix('v2_').removesuffix('_share')
                                   for name in self.activity_names]
            self.feature_names += ['activity_other_fraction', 'activity_present']

        def pairs(self, g, panel, candidates):
            sizes = np.asarray([len(row) for row in candidates], dtype=np.int32)
            q = np.repeat(np.arange(len(g)), sizes)
            products = np.concatenate(candidates)
            values = panel[q, products]
            identity = np.column_stack([(self.catalog[products] == p) & (int(p) in self.id_allowed)
                                        for p in self.codes])
            base = np.column_stack([g[q, :8], values[:, :29], values[:, 29:34], identity]).astype(np.float32)
            present = (g[:, 0] > 0) & (g[:, 2] < 1)
            fractions = np.clip(g[:, 8:] / np.maximum(1 - g[:, 2, None], 1e-6), 0, 1)
            other = np.where(present, np.maximum(1 - fractions.sum(axis=1), 0), 0)
            matrix = np.column_stack([base, fractions[q], other[q], present[q]]).astype(np.float32)
            assert matrix.shape[1] == len(self.feature_names) and np.isfinite(matrix).all()
            return csr_matrix(matrix), sizes

    def candidate_rows(data, schema, panel, labels, pool, rng, negatives=4, mode="uniform"):
        pool_indices = np.asarray([data.catalog_index[int(p)] for p in pool])
        result = []
        for row, label in enumerate(labels):
            positive = data.catalog_index[int(label)]
            available = pool_indices[pool_indices != positive]
            k = min(negatives, len(available))
            if mode == "uniform":
                chosen = rng.choice(available, k, replace=False)
            else:
                hardness = 2 * np.log1p(panel[row, available, 1]) + np.log1p(panel[row, available, 0])
                hardness += 4 * panel[row, available, BASE_SUFFIXES.index("interest_decayed_target_share")]
                order = np.argsort(-(hardness + rng.uniform(0, 1e-6, len(available))))
                hard = available[order[:k // 2]]
                rest = available[~np.isin(available, hard)]
                chosen = np.r_[hard, rng.choice(rest, k - len(hard), replace=False)]
            candidates = rng.permutation(np.r_[positive, chosen]).astype(np.int32)
            assert len(np.unique(candidates)) == len(candidates) and (schema.catalog[candidates] == label).sum() == 1
            result.append(candidates)
        return result

    def matrix_cache(data, schema, rows, split, spec, folder, training):
        key = joblib.hash((data.store["signature"], schema.feature_names, schema.source, rows,
                           split["pool"], spec.get("negatives", 4), spec.get("negative_mode", "uniform"), training))
        if spec.get("negative_seed", CANONICAL_SEED) != CANONICAL_SEED:
            key = joblib.hash((key, spec["negative_seed"]))
        path = folder / f"matrix_{key}.joblib"
        if path.exists():
            result = joblib.load(path, mmap_mode="r")
            assert result["key"] == key
            return result
        stem = folder / f"building_{key}"
        data_path, index_path, pointer_path = [Path(str(stem) + suffix) for suffix in (".data", ".indices", ".indptr")]
        target_blocks, group_blocks = [], []
        nnz = n_rows = 0
        rng = np.random.default_rng(spec.get("negative_seed", CANONICAL_SEED))
        with data_path.open("xb") as fd, index_path.open("xb") as fi, pointer_path.open("xb") as fp:
            fp.write(np.asarray([0], dtype=np.int32).tobytes())
            for start in range(0, len(rows), BATCH):
                batch_rows = rows[start:start + BATCH]
                g, panel = schema.panel(data, batch_rows)
                labels = data.y[batch_rows]
                if training:
                    candidates = candidate_rows(data, schema, panel, labels, split["pool"], rng,
                                                 spec.get("negatives", 4), spec.get("negative_mode", "uniform"))
                else:
                    candidates = [np.asarray([data.catalog_index[int(p)] for p in split["pool"]]) for _ in batch_rows]
                matrix, groups = schema.pairs(g, panel, candidates)
                targets = np.concatenate([(schema.catalog[c] == label).astype(np.int32) for c, label in zip(candidates, labels)])
                assert all((schema.catalog[c] == label).sum() == 1 for c, label in zip(candidates, labels))
                fd.write(matrix.data.astype(np.float32).tobytes())
                fi.write(matrix.indices.astype(np.int32).tobytes())
                fp.write((matrix.indptr[1:] + nnz).astype(np.int32).tobytes())
                nnz += matrix.nnz
                n_rows += matrix.shape[0]
                target_blocks.append(targets)
                group_blocks.append(groups)
                del matrix, g, panel
                data.release()
        values = np.memmap(data_path, dtype=np.float32, mode="r", shape=(nnz,))
        indices = np.memmap(index_path, dtype=np.int32, mode="r", shape=(nnz,))
        pointers = np.memmap(pointer_path, dtype=np.int32, mode="r", shape=(n_rows + 1,))
        matrix = csr_matrix((values, indices, pointers), shape=(n_rows, len(schema.feature_names)), copy=False)
        payload = dict(key=key, matrix=matrix, target=np.concatenate(target_blocks), groups=np.concatenate(group_blocks))
        with path.open("xb") as f:
            joblib.dump(payload, f, compress=0)
        del payload, matrix, values, indices, pointers
        for tmp in (data_path, index_path, pointer_path):
            tmp.unlink()
        gc.collect()
        return joblib.load(path, mmap_mode="r")

    def raw_scores(data, schema, model, rows, test=False, iteration=None):
        scores = np.empty((len(rows), len(data.codes)), dtype=np.float32)
        candidates = np.asarray([data.catalog_index[int(p)] for p in data.codes])
        for start in range(0, len(rows), BATCH):
            r = rows[start:start + BATCH]
            g, panel = schema.panel(data, r, test=test)
            pairs, _ = schema.pairs(g, panel, [candidates for _ in r])
            options = dict(num_iteration=iteration or model.best_iteration_ or model.n_estimators_)
            if isinstance(model, lgb.LGBMClassifier):
                options["raw_score"] = True
            scores[start:start + len(r)] = model.predict(pairs, **options).reshape(len(r), -1)
            data.release()
        assert np.isfinite(scores).all()
        return scores

    def ranker_temperature(scores, labels, codes):
        target = np.searchsorted(codes, labels)
        def loss(log_t):
            z = scores / np.exp(log_t)
            return float((logsumexp(z, axis=1) - z[np.arange(len(z)), target]).mean())
        fitted = minimize_scalar(loss, bounds=(math.log(0.1), math.log(10)), method="bounded")
        return float(np.exp(fitted.x))

    def probabilities(scores, temp):
        z = scores.astype(np.float64) / temp
        z -= z.max(axis=1, keepdims=True)
        z = np.exp(z)
        return (z / z.sum(axis=1, keepdims=True)).astype(np.float32)


In [ ]:
if RUN_LGBM_ENSEMBLE and RANKER_SOURCE != 'bridge':
    import inspect, textwrap
    from sklearn.feature_extraction import DictVectorizer

    def make_ranker_feature_functions(event_dictionary):
        from collections import Counter
        TIME_SCALE_SECONDS = 0.001
        event_types = event_dictionary.set_index('encoded_event')['event_type'].to_dict()
        target_event_codes = set(event_dictionary.loc[event_dictionary['is_target_event'].fillna(False), 'encoded_event'])
        action_types = sorted(set(event_types.values()) | {'unknown'})

        def time_features(timestamps):
            known = timestamps[np.isfinite(timestamps)]
            gaps = np.diff(timestamps)
            gaps = gaps[np.isfinite(gaps)] * TIME_SCALE_SECONDS
            return {
                'duration_hours': (known[-1] - known[0]) * TIME_SCALE_SECONDS / 3600 if len(known) > 1 else 0, # общее время истории
                'n_missing_timestamps': len(timestamps) - len(known),                                          # события без времени
                'gap_mean_seconds': gaps.mean() if len(gaps) else 0,                                           # средний промежуток
                'gap_median_seconds': np.median(gaps) if len(gaps) else 0,                                     # мед. пром
                'gap_p90_seconds': np.quantile(gaps, 0.9) if len(gaps) else 0,                                 # 90-ый перц
                'gap_max_seconds': gaps.max() if len(gaps) else 0,                                             # максимальный пром.
            }

        def history_features(events):
            missing = events is None
            events = events or []
            kinds = [event_types.get(e['encoded_event'], 'unknown') if e else 'unknown' for e in events]
            products = [e['encoded_product'] if e else None for e in events]
            target_flags = [e is not None and e['encoded_event'] in target_event_codes for e in events]
            timestamps = np.asarray([e['timestamp'] if e else None for e in events], dtype=np.float64)

            known_times = timestamps[np.isfinite(timestamps)]
            if (np.diff(known_times) < 0).any():
                raise ValueError('Нарушен хронологический порядок событий.')

            end_time = known_times[-1] if len(known_times) else np.nan
            known = [p for p in products if p is not None]
            product_counts = Counter(known)
            counts = Counter(kinds)
            target_products = [p for p, flag in zip(products, target_flags) if flag]
            target_counts = Counter(p for p in target_products if p is not None)
            last_target_indices = {p: i for i, (p, flag) in enumerate(zip(products, target_flags))
                                   if flag and p is not None}
            n = len(events)

            # Условные сессии отделяем паузой 30 минут; неизвестное время не создаёт новую сессию.
            session_ids = np.zeros(n, dtype=np.int32)
            previous_time, session = None, 0
            for i, timestamp in enumerate(timestamps):
                if np.isfinite(timestamp):
                    if previous_time is not None and (timestamp - previous_time) * TIME_SCALE_SECONDS > 1800:
                        session += 1
                    previous_time = timestamp
                session_ids[i] = session
            last_session = session_ids[-1] if n else -1
            session_clicks, session_targets = Counter(), Counter()
            click_sessions, target_sessions = {}, {}
            decayed_clicks, decayed_targets = Counter(), Counter()
            for i, (product, kind, flag) in enumerate(zip(products, kinds, target_flags)):
                if product is None:
                    continue
                if kind == 'click':
                    click_sessions.setdefault(product, set()).add(int(session_ids[i]))
                    session_clicks[product] += int(session_ids[i] == last_session)
                if flag:
                    target_sessions.setdefault(product, set()).add(int(session_ids[i]))
                    session_targets[product] += int(session_ids[i] == last_session)
                if np.isfinite(timestamps[i]) and np.isfinite(end_time):
                    # Вес интереса уменьшается вдвое за сутки, считая от конца доступной истории.
                    weight = np.exp2(-(end_time - timestamps[i]) * TIME_SCALE_SECONDS / 86400)
                    if kind == 'click': decayed_clicks[product] += weight
                    if flag: decayed_targets[product] += weight
            recent_interest = {}

            # Общие признаки истории
            result = {
                'history_length': n, 'history_missing': int(missing),                                      # кол-во событий и отсутствие истории
                'n_product_events': len(known), 'known_product_rate': len(known) / max(n, 1),              # кол-во и доля событий с продуктом
                'n_unique_products': len(product_counts),                                                  # кол-во разных продуктов
                'last_product': str(known[-1]) if known else 'missing',                                    # последний известный продукт
                'most_frequent_product': str(min(product_counts, key=lambda p: (-product_counts[p], p)))   # самый частый продукт
                                         if product_counts else 'missing',
            }

            # Сессионная активность дополняет общие счётчики
            result.update({'interest_n_sessions': int(session + 1) if n else 0,                            # кол-во сессий
                           'interest_last_session_length': int((session_ids == last_session).sum()),       # кол-во событий в последней сессии
                           'interest_last_session_n_click': sum(kind == 'click' and sid == last_session    # кол-во кликов в последней сессии
                                                                for kind, sid in zip(kinds, session_ids)),
                           'interest_last_session_n_target': sum(flag and sid == last_session              # кол-во целевых действий в последней сессии
                                                                 for flag, sid in zip(target_flags, session_ids))})

            result.update({'n_target_events': len(target_products),                                        # кол-во событий целевого типа
                           'last_target_event_product': str(target_products[-1])
                           if target_products and target_products[-1] is not None else 'missing'})         # продукт последнего такого события

            result.update({f'history_{key}': value for key, value in time_features(timestamps).items()})   # добавляем все временные признаки

            # Количество и доля каждого типа действия
            for kind in action_types:
                result[f'n_{kind}'] = counts[kind]
                result[f'{kind}_rate'] = counts[kind] / max(n, 1)

            # Продукты последних просмотра и клика
            for kind in ['click', 'view']:
                interacted = [p for p, action in zip(products, kinds) if action == kind]
                result[f'last_{kind}_product'] = str(interacted[-1]) if interacted and interacted[-1] is not None else 'missing'

            # Признаки по последним 20 и 100 событиям
            for window in [20, 100]:
                tail_kinds, tail_products = kinds[-window:], products[-window:]
                tail_known = [p for p in tail_products if p is not None]
                tail_counts = Counter(tail_kinds)
                size = len(tail_kinds)
                prefix = f'last{window}_'

                result.update({prefix + 'n_events': size,                                                  # кол-во последних событий
                               prefix + 'n_unique_products': len(set(tail_known)),                         # кол-во разных продуктов
                               prefix + 'known_product_rate': len(tail_known) / max(size, 1)})             # доля событий с продуктом

                result.update({prefix + key: value for key, value in time_features(timestamps[-window:]).items()})  # временные хар-ки

                for kind in action_types:
                    result[prefix + f'n_{kind}'] = tail_counts[kind]                                       # кол-во действий данного типа
                    result[prefix + f'{kind}_rate'] = tail_counts[kind] / max(size, 1)                     # доля действий данного типа

                window_product_counts = {}

                # Просмотры и клики по каждому продукту в последних событиях
                for kind in ['view', 'click']:
                    product_counts_window = Counter(p for p, action in zip(tail_products, tail_kinds)
                                                    if p is not None and action == kind)
                    window_product_counts[kind] = product_counts_window
                    result.update({f'p_{product}_last{window}_{kind}': count
                                   for product, count in product_counts_window.items()})

                # События целевого типа по продуктам в последних 20/100 событиях
                target_window_counts = Counter(p for p, flag in zip(tail_products, target_flags[-window:])
                                               if flag and p is not None)
                result.update({f'p_{product}_last{window}_target_event': count
                               for product, count in target_window_counts.items()})

                recent_interest[window] = {**window_product_counts, 'target': target_window_counts, 'size': size}

            clicks = Counter(p for p, kind in zip(products, kinds) if kind == 'click' and p is not None)
            views = Counter(p for p, kind in zip(products, kinds) if kind == 'view' and p is not None)
            recent = Counter(p for p in products[-100:] if p is not None)
            last_indices = {p: i for i, p in enumerate(products) if p is not None}

            # Признаки отдельно для каждого продукта
            for product, count in product_counts.items():
                last_index = last_indices[product]
                time_known = np.isfinite(timestamps[last_index]) and np.isfinite(end_time)
                prefix = f'p_{product}_'

                result.update({
                    prefix + 'count': count,                                                               # кол-во взаимодействий с продуктом
                    prefix + 'share': count / n,                                                           # доля продукта среди всех событий
                    prefix + 'click': clicks[product],                                                     # кол-во кликов
                    prefix + 'view': views[product],                                                       # кол-во просмотров
                    prefix + 'last100': recent[product],                                                   # упоминания в последних 100 событиях
                    prefix + 'last_position': (last_index + 1) / n,                                        # положение последнего взаимодействия
                    prefix + 'hours_since_last': (end_time - timestamps[last_index]) * TIME_SCALE_SECONDS / 3600
                                                 if time_known else 0,                                     # часов с последнего взаимодействия
                    prefix + 'last_time_missing': int(not time_known)                                      # отсутствует время взаимодействия
                })

                last_target_index = last_target_indices.get(product)
                target_time_known = (last_target_index is not None and np.isfinite(timestamps[last_target_index])
                                     and np.isfinite(end_time))

                result.update({
                    prefix + 'target_event_count': target_counts[product],                                 # кол-во событий целевого типа
                    prefix + 'hours_since_last_target_event':
                        (end_time - timestamps[last_target_index]) * TIME_SCALE_SECONDS / 3600
                        if target_time_known else 0,                                                       # часов с последнего такого события
                    prefix + 'last_target_event_time_missing': int(not target_time_known)                  # нет времени такого события
                })

                # Пять добавочных просмотров сглаживают отношения при маленьких счётчиках
                result.update({prefix + 'interest_click_view_ratio': clicks[product] / (views[product] + 5),                 # отношение кликов к просмотрам
                               prefix + 'interest_target_view_ratio': target_counts[product] / (views[product] + 5),         # отношение целевых действий к просмотрам
                               prefix + 'interest_click_share': clicks[product] / max(counts['click'], 1),                   # доля продукта среди всех кликов
                               prefix + 'interest_target_share': target_counts[product] / max(len(target_products), 1)})     # доля продукта среди целевых действий

                for window in [20, 100]:
                    window_counts = recent_interest[window]
                    result[prefix + f'interest_last{window}_click_view_ratio'] = window_counts['click'][product] / (window_counts['view'][product] + 5)    # отношение кликов к просмотрам в окне
                    result[prefix + f'interest_last{window}_target_view_ratio'] = window_counts['target'][product] / (window_counts['view'][product] + 5)  # отношение целевых действий к просмотрам в окне

                # Сравниваем последние 20 событий с предыдущими 80 и недавний интерес со всей историей
                r20, r100 = recent_interest[20], recent_interest[100]
                for action in ['click', 'target']:
                    total = clicks[product] if action == 'click' else target_counts[product]
                    result[prefix + f'interest_recent_{action}_fraction'] = r100[action][product] / max(total, 1)  # доля действий продукта в последних 100 событиях
                    result[prefix + f'interest_{action}_growth'] = (r20[action][product] / max(r20['size'], 1)     # изменение частоты действий в последних 20 событиях
                        - (r100[action][product] - r20[action][product]) / max(r100['size'] - r20['size'], 1))

                # Доли с затуханием и повторение интереса в разных сессиях меньше зависят от числа показов
                result.update({prefix + 'interest_decayed_click_share': decayed_clicks[product] / (sum(decayed_clicks.values()) or 1.0),                   # доля кликов с уменьшением веса старых
                               prefix + 'interest_decayed_target_share': decayed_targets[product] / (sum(decayed_targets.values()) or 1.0),                # доля целевых действий с уменьшением веса старых
                               prefix + 'interest_n_click_sessions': len(click_sessions.get(product, ())),                                                 # кол-во сессий с кликами по продукту
                               prefix + 'interest_n_target_sessions': len(target_sessions.get(product, ())),                                               # кол-во сессий с целевыми действиями
                               prefix + 'interest_last_session_click': session_clicks[product],                                                            # кол-во кликов по продукту в последней сессии
                               prefix + 'interest_last_session_target': session_targets[product],                                                          # кол-во целевых действий в последней сессии
                               prefix + 'interest_last_session_click_share': session_clicks[product] / max(sum(session_clicks.values()), 1),               # доля кликов продукта в последней сессии
                               prefix + 'interest_last_session_target_share': session_targets[product] / max(sum(session_targets.values()), 1)})           # доля целевых действий продукта в последней сессии

            # Временные признаки относительно конца доступной истории
            days = np.where(np.isfinite(timestamps) & np.isfinite(end_time),
                            (end_time - timestamps) * TIME_SCALE_SECONDS / 86400, np.nan)
            result.update({
                'v2_end_timestamp_ms': end_time if np.isfinite(end_time) else -1,                                  # время конца истории
                'v2_end_time_missing': int(not n or not np.isfinite(timestamps[-1])),                              # отсутствие времени последнего события
                'v2_n_active_days': len(set((known_times * TIME_SCALE_SECONDS // 86400).astype(np.int64))),        # кол-во активных дней
            })

            event_counts, recent_event_counts = Counter(), Counter()
            positions, timed_actions = {}, {}
            transitions, after_target = Counter(), Counter()
            previous_target = None
            last_target_position = max((i for i, flag in enumerate(target_flags) if flag), default=-1)
            last_target_product = products[last_target_position] if last_target_position >= 0 else None
            timed_counts = {(action, window): Counter() for action in ['click', 'view', 'target'] for window in [1, 3, 7]}
            previous_week = {'click': Counter(), 'target': Counter()}
            global_timed_counts = Counter()
            product_days, product_sessions = {}, {}

            # Один проход: календарные окна, возвраты и переходы между целевыми действиями
            for i, (product, kind, flag) in enumerate(zip(products, kinds, target_flags)):
                if events[i] is not None and kind != 'view':
                    event_code = events[i]['encoded_event']
                    event_counts[event_code] += 1
                    if i >= n - 100:
                        recent_event_counts[event_code] += 1
                if np.isfinite(days[i]):
                    for window in [1, 3, 7]:
                        if days[i] <= window:
                            global_timed_counts['click', window] += int(kind == 'click')
                            global_timed_counts['target', window] += int(flag)
                if product is None:
                    if flag:
                        previous_target = None
                    continue
                product_sessions.setdefault(product, set()).add(int(session_ids[i]))
                if np.isfinite(timestamps[i]):
                    product_days.setdefault(product, set()).add(int(timestamps[i] * TIME_SCALE_SECONDS // 86400))
                if i > last_target_position >= 0:
                    after_target[product] += 1
                actions = ([kind] if kind in ['click', 'view'] else []) + (['target'] if flag else [])
                for action in actions:
                    key = (product, action)
                    positions.setdefault(key, []).append(i)
                    if np.isfinite(timestamps[i]):
                        timed_actions.setdefault(key, []).append(timestamps[i])
                    if np.isfinite(days[i]):
                        for window in [1, 3, 7]:
                            if days[i] <= window:
                                timed_counts[action, window][product] += 1
                        if action != 'view' and 1 < days[i] <= 7:
                            previous_week[action][product] += 1
                if flag:
                    if previous_target is not None:
                        transitions[previous_target, product] += 1
                    previous_target = product

            # Коды действий сохраняют информацию, которая теряется при объединении в view/click/other
            for event_code, count in event_counts.items():
                result[f'v2_event_{event_code}_share'] = count / max(n, 1)                                                      # доля событий данного кода
                result[f'v2_event_{event_code}_last100_share'] = recent_event_counts[event_code] / max(min(n, 100), 1)          # доля кода в последних 100 событиях
            result['v2_events_after_last_target'] = n - last_target_position - 1 if last_target_position >= 0 else n            # кол-во событий после последнего целевого
            result['v2_has_target'] = int(last_target_position >= 0)                                                            # наличие целевых действий в истории
            result['v2_target_product_switches'] = sum(count for (left, right), count in transitions.items() if left != right)  # кол-во смен продукта между целевыми действиями

            for window in [1, 3, 7]:
                result[f'v2_click_{window}d'] = global_timed_counts['click', window]                                            # кол-во кликов за последние дни
                result[f'v2_target_{window}d'] = global_timed_counts['target', window]                                          # кол-во целевых действий за последние дни

            total_windows = {key: sum(value.values()) for key, value in timed_counts.items()}
            rank_values = {(product, action): sum(value > counter[product] for value in counter.values()) / max(len(product_counts) - 1, 1)
                           for action, counter in [('click', clicks), ('target', target_counts)] for product in product_counts}

            # Ранги отражают интерес внутри истории, а не числовые значения кодов продуктов
            for product in product_counts:
                prefix = f'p_{product}_v2_'
                result.update({
                    prefix + 'seen': 1,                                                                                                  # наличие продукта в истории
                    prefix + 'n_active_days': len(product_days.get(product, ())),                                                        # кол-во дней с продуктом
                    prefix + 'session_fraction': len(product_sessions.get(product, ())) / max(session + 1, 1),                           # доля сессий с продуктом
                    prefix + 'sessions_since_last': max(last_session - max(product_sessions.get(product, {-1})), 0),                     # кол-во сессий с последнего взаимодействия
                    prefix + 'follow_last_target': transitions[last_target_product, product] if last_target_product is not None else 0,  # переходы от последнего целевого продукта
                    prefix + 'after_last_target_share': after_target[product] / max(sum(after_target.values()), 1),                      # доля событий продукта после последнего целевого
                })

                for action in ['click', 'view', 'target']:
                    indices = positions.get((product, action), [])
                    times = timed_actions.get((product, action), [])
                    time_ok = bool(indices) and np.isfinite(timestamps[indices[-1]]) and np.isfinite(end_time)
                    result[prefix + f'has_{action}'] = int(bool(indices))                                                                          # наличие действия данного типа с продуктом
                    result[prefix + f'{action}_time_missing'] = int(not time_ok)                                                                   # отсутствие времени последнего действия
                    result[prefix + f'hours_since_{action}'] = (end_time - timestamps[indices[-1]]) * TIME_SCALE_SECONDS / 3600 if time_ok else 0  # часов с последнего действия
                    result[prefix + f'steps_since_{action}'] = (n - 1 - indices[-1]) / max(n, 1) if indices else 1                                 # доля событий после последнего действия

                    for window in [1, 3, 7]:
                        count = timed_counts[action, window][product]
                        result[prefix + f'{action}_{window}d'] = count                                                                             # кол-во действий продукта за последние дни
                        result[prefix + f'{action}_{window}d_share'] = count / max(total_windows[action, window], 1)                               # доля продукта среди действий за последние дни

                    if action in ['click', 'target']:
                        gaps = np.diff(times) * TIME_SCALE_SECONDS / 3600
                        second_ok = len(indices) > 1 and np.isfinite(timestamps[indices[-2]]) and np.isfinite(end_time)
                        result[prefix + f'{action}_rank'] = rank_values[product, action]                                                                        # относительный ранг продукта по числу действий
                        result[prefix + f'{action}_acceleration'] = timed_counts[action, 1][product] - previous_week[action][product] / 6                       # активность за сутки минус средняя за 6 дней
                        result[prefix + f'{action}_last2_session_share'] = sum(session_ids[j] >= last_session - 1 for j in indices) / max(len(indices), 1)      # доля действий в последних двух сессиях
                        result[prefix + f'hours_since_previous_{action}'] = (end_time - timestamps[indices[-2]]) * TIME_SCALE_SECONDS / 3600 if second_ok else 0# часов с предпоследнего действия
                        result[prefix + f'previous_{action}_time_missing'] = int(not second_ok)                                                                 # отсутствие времени предпоследнего действия
                        result[prefix + f'{action}_gap_mean_hours'] = gaps.mean() if len(gaps) else 0                                                           # средний интервал между действиями в часах
                        result[prefix + f'{action}_gap_last_hours'] = gaps[-1] if len(gaps) else 0                                                              # последний интервал между действиями в часах
                        result[prefix + f'{action}_gap_count'] = len(gaps)                                                                                      # кол-во интервалов с известным временем

            return result

        return history_features, time_features


    def ranker_feature_rows(path, expected_ids):
        offset = 0
        for batch in pq.ParquetFile(path).iter_batches(
                batch_size=32, columns=['example_id', 'events'], use_threads=False):
            for i in range(len(batch)):
                assert batch['example_id'][i].as_py() == expected_ids[offset + i], 'Нарушен порядок историй.'
                features = ranker_history_function(batch['events'][i].as_py())
                yield {name: value for name, value in features.items() if isinstance(value, str) or value != 0}
            offset += len(batch)
        assert offset == len(expected_ids)


    def prepare_ranker_aggregates():
        candidates = ([Path(RANKER_CACHE_FILE)] if RANKER_CACHE_FILE is not None else
                      sorted((WORK / 'ranker_features').glob('aggregates_raw_*.joblib'), reverse=True)
                      + sorted(INPUT_ROOT.rglob('aggregates.joblib')))
        if USE_RANKER_FEATURE_CACHE:
            for path in candidates:
                if not path.is_file():
                    continue
                cached = None
                try:
                    cached = joblib.load(path, mmap_mode='r')
                    legacy = (cached.get('signature', {}).get('code')
                              == '38fa868c9b8d9f2360468ed2ffd4c7adceb4c96b3882088eb3acd78182511284')
                    if legacy:
                        compatible = sha256_file(path) == '06f9e5048e27ae94e0094bdb447908d8e3bd7bbee2baee0870e5018539ed6a75'
                    else:
                        compatible = cached.get('signature') == ranker_generator_signature
                    if not compatible:
                        raise ValueError('Несовместимая сигнатура кэша.')
                    assert cached['train_ids'] == train_meta.example_id.tolist()
                    assert cached['test_ids'] == test_meta.example_id.tolist()
                    assert joblib.hash(cached['names']) == 'd7bd07e8a758a8419ad38182d6466134'
                    for split, size in [('train', len(train_meta)), ('test', len(test_meta))]:
                        matrix = cached[split]
                        assert matrix.shape == (size, 6223) and np.isfinite(matrix.data).all()
                except Exception as error:
                    # Необязательный кэш не должен блокировать расчёт из Parquet.
                    del cached
                    print('Кэш пропущен:', path.name, '|', type(error).__name__)
                    continue
                print('Загружен', 'исторический' if legacy else 'рассчитанный из Parquet', 'кэш:', path)
                return cached

        # Словарь столбцов строим только на train; test использует тот же порядок.
        vectorizer = DictVectorizer(sparse=True, dtype=np.float32)
        train_features = vectorizer.fit_transform(ranker_feature_rows(
            DATA_DIR / 'train.parquet', train_meta.example_id.tolist()))
        test_features = vectorizer.transform(ranker_feature_rows(
            DATA_DIR / 'test.parquet', test_meta.example_id.tolist()))
        names = vectorizer.get_feature_names_out()
        assert len(names) == 6223 and joblib.hash(names) == 'd7bd07e8a758a8419ad38182d6466134', (
            'Состав или порядок признаков отличается от исторической схемы.')
        for matrix in [train_features, test_features]:
            matrix.eliminate_zeros()
            assert np.isfinite(matrix.data).all()
        assert train_features.shape == (len(train_meta), len(names))
        assert test_features.shape == (len(test_meta), len(names))
        store = dict(signature=ranker_generator_signature, train=train_features, test=test_features,
                     names=names, train_ids=train_meta.example_id.tolist(), test_ids=test_meta.example_id.tolist())
        if SAVE_RANKER_FEATURE_CACHE:
            folder = WORK / 'ranker_features'
            folder.mkdir(parents=True, exist_ok=True)
            path = folder / f'aggregates_raw_{time.time_ns()}.joblib'
            with path.open('xb') as stream:
                joblib.dump(store, stream, compress=0)
            del store, train_features, test_features, vectorizer, matrix
            gc.collect()
            print('Сохранён новый кэш с собственной сигнатурой:', path)
            return joblib.load(path, mmap_mode='r')
        return store


### Признаки и разбиение

Генератор перенесён из раздела 3 третьего ноутбука без изменения формул. Его словарь событий изолирован от переменных GRU. В pandas не переносим вложенные истории: обрабатываем одну историю внутри небольшого пакета PyArrow и собираем разреженные числовые матрицы. Словарь признаков строим на train, test преобразуем тем же словарём.

Перед переносом сравнили 112 историй train и 64 истории test из начала, середины и конца файлов: все 6223 столбца совпали с историческим кэшем побайтово, ID и порядок строк совпали. Два новых признака кода события в test отсутствуют в словаре train и игнорируются, как в исходном преобразовании. Для обеих схем Ranker дополнительно совпали все пары с 32 кандидатами на 24 историях train и 24 историях test.

Текстовые хеши генераторов различаются. Исходный текст исторической версии не найден, поэтому строго доказать, что различия только в комментариях, нельзя. Проверка значений на выборке пройдена; полное сравнение всех строк ещё не выполнялось. Новый кэш сохраняет фактический хеш текущего генератора, хеши исходных файлов и словаря — историческая сигнатура не подставляется.

Исторический кэш принимаем только по его известному SHA256; новый — при совпадении собственной сигнатуры. В обоих случаях проверяем ID, размеры и порядок 6223 столбцов. Старые файлы не перезаписываем. Шестнадцать кодов непросмотровой активности для второй модели выбираем только по fit.


In [ ]:
if RUN_LGBM_ENSEMBLE and RANKER_SOURCE != 'bridge':
    if SMOKE_TEST:
        raise RuntimeError('Раздел 9 требует полного набора данных и обученной GRU.')
    if RANKER_SOURCE not in ['saved', 'train']:
        raise ValueError("RANKER_SOURCE: только 'saved' или 'train'.")

    ranker_source_hashes = {
        'train.parquet': 'e23da11fbd067a512448ac2c95929dceb368dfd0b36d941071588b23527c50dd',
        'test.parquet': '7ee30224a75892d458ed4ce539e4c34445b192aecdd14b9ce034bf4e719952d5',
        'event_dictionary.parquet': 'c776e0ed82c89bd7388c910d95b2bb142b3a8af3c71b62219d7a7e0a63ada9b9'
    }
    assert all(source_hashes[name] == digest for name, digest in ranker_source_hashes.items())
    ranker_history_function, ranker_time_function = make_ranker_feature_functions(dictionary)
    ranker_generator_signature = dict(
        version='v4_temporal_interest_20261110', time_scale=0.001,
        code=hashlib.sha256((textwrap.dedent(inspect.getsource(ranker_history_function))
                            + textwrap.dedent(inspect.getsource(ranker_time_function))).encode()).hexdigest(),
        pipeline_code=hashlib.sha256(''.join(textwrap.dedent(inspect.getsource(function))
            for function in [make_ranker_feature_functions, ranker_feature_rows, prepare_ranker_aggregates]).encode()).hexdigest(),
        sources_sha256={name: source_hashes[name] for name in ranker_source_hashes},
        dictionary=hashlib.sha256(pd.util.hash_pandas_object(dictionary, index=True).values.tobytes()).hexdigest()
    )
    ranker_store = prepare_ranker_aggregates()
    ranker_data = RankerData(ranker_store)
    assert np.array_equal(ranker_data.codes, classes)
    ranker_split = dict(fit=train_idx, valid=valid_idx, es=es_idx, outer=outer_idx,
                        pool=np.sort(np.unique(ranker_data.y[train_idx])))
    assert np.array_equal(ranker_split['pool'], classes)
    for name, rows in [('train_idx', train_idx), ('valid_idx', valid_idx),
                       ('es_idx', es_idx), ('outer_idx', outer_idx)]:
        assert hashlib.sha256(np.asarray(rows, dtype='<i8').tobytes()).hexdigest() == expected_split_hashes[name]
    for a, b in [(train_idx, valid_idx), (es_idx, outer_idx)]:
        assert not set(train_meta.iloc[a].client_id) & set(train_meta.iloc[b].client_id)
    ranker_schemas = {'old': Schema(ranker_data, ranker_split),
                      'activity': ActivitySchema(ranker_data, ranker_split)}
    assert [len(schema.feature_names) for schema in ranker_schemas.values()] == [74, 92]
    assert ranker_schemas['activity'].feature_names[:74] == ranker_schemas['old'].feature_names
    assert ranker_schemas['activity'].activity_names == [
        f'v2_event_{code}_share' for code in [99, 687, 234, 1062, 959, 774, 971, 820,
                                            842, 125, 299, 622, 82, 91, 80, 343]]
    print('Ranker: проверены кэш, 74/92 признака, классы, ID и fit / ES / outer.')


### Обучение и вероятности LightGBM

У первого Ranker фиксированы 493 дерева; у `activity_profile` — максимум 500 и early stopping 80 по NDCG@1 на ES. Для каждой истории оставляем правильный продукт и четыре случайных отрицательных кандидата, seed 42. Температуры каждой модели подбираются только по log loss на ES; затем усредняем вероятности 50/50.

На исторических весах ожидаются температуры 0.9100674248099417 и 0.9012745242629806, а у второй модели — 393 дерева. При новом обучении совпадение проверяется отдельно: версии библиотек и численные вычисления могут повлиять на результат. В артефакте записаны LightGBM 4.7.0 и NumPy 2.5.4.

Ранее выполненная лёгкая проверка переноса: схемы и первые 256 историй обучающих пар совпали с сохранёнными матрицами. Оценки обеих моделей на 128 историях validation и 128 историях test совпали побайтово; калибровка всех сохранённых score-массивов восстановила весь bridge побайтово. Новое обучение не запускалось.


In [ ]:
if RUN_LGBM_ENSEMBLE and RANKER_SOURCE != 'bridge':
    ranker_models = {}
    if RANKER_SOURCE == 'saved':
        ranker_bundle_hash = '9c69ab9074c6bdc76cc977c6b7c977519cfff1e831b2990443af127bafdae00b'
        ranker_bundle_candidates = ([Path(RANKER_BUNDLE_FILE)] if RANKER_BUNDLE_FILE is not None
                                    else sorted(INPUT_ROOT.rglob('selected_ensemble.joblib')))
        ranker_bundle_matches = [p for p in ranker_bundle_candidates
                                 if p.is_file() and sha256_file(p) == ranker_bundle_hash]
        if not ranker_bundle_matches:
            raise FileNotFoundError("Нет исторического selected_ensemble.joblib; для обучения выберите 'train'.")
        ranker_bundle = joblib.load(ranker_bundle_matches[0])
        assert ranker_bundle['data_signature'] == ranker_data.store['signature'], ('Сигнатура отличается от исторических весов; для нового расчёта используйте RANKER_SOURCE=\'train\'.')
        assert np.array_equal(ranker_bundle['codes'], classes)
        assert np.array_equal(ranker_bundle['fit_indices'], train_idx)
        for name, schema in ranker_schemas.items():
            state = ranker_bundle['schema_states'][name]
            assert schema.feature_names == state['feature_names']
            assert np.array_equal(schema.source, state['source'])
            assert np.array_equal(schema.globals, state['globals'])
            assert schema.id_allowed == state['id_allowed']
            ranker_models[name] = ranker_bundle['models'][name]
            assert ranker_models[name].n_features_in_ == len(schema.feature_names)
    else:
        # Уникальный каталог исключает перезапись исторических моделей и матриц.
        ranker_run = WORK / 'ranker_037191' / str(time.time_ns())
        ranker_run.mkdir(parents=True, exist_ok=False)
        ranker_spec = dict(negatives=4, negative_mode='uniform')
        for name, schema in ranker_schemas.items():
            ranker_matrix_dir = ranker_run / name
            ranker_matrix_dir.mkdir()
            ranker_fit = matrix_cache(ranker_data, schema, train_idx, ranker_split,
                                       ranker_spec, ranker_matrix_dir, training=True)
            ranker_params = dict(BASE_PARAMS)
            ranker_kwargs = dict(group=ranker_fit['groups'], callbacks=[lgb.log_evaluation(100)])
            if name == 'old':
                ranker_params['n_estimators'] = 493
            else:
                ranker_es = matrix_cache(ranker_data, schema, es_idx, ranker_split,
                                          ranker_spec, ranker_matrix_dir, training=False)
                ranker_kwargs.update(categorical_feature=[], eval_set=[(ranker_es['matrix'], ranker_es['target'])],
                                      eval_group=[ranker_es['groups']], eval_at=[1])
                ranker_kwargs['callbacks'].append(lgb.early_stopping(80, first_metric_only=True, verbose=False))
            ranker_model = lgb.LGBMRanker(**ranker_params)
            ranker_started = perf_counter()
            ranker_model.fit(ranker_fit['matrix'], ranker_fit['target'], **ranker_kwargs)
            ranker_models[name] = ranker_model
            print(name, '| деревьев:', ranker_model.best_iteration_ or ranker_model.n_estimators_,
                  '| секунд:', round(perf_counter() - ranker_started, 1))
            with (ranker_run / f'{name}.joblib').open('xb') as stream:
                joblib.dump(dict(model=ranker_model, schema_state=schema.__dict__, params=ranker_params,
                                 split=ranker_split, codes=classes, data_signature=ranker_data.store['signature']), stream)
            del ranker_fit
            if name == 'activity':
                del ranker_es
            gc.collect()
            ranker_data.release()

    ranker_valid_probability = np.zeros((len(valid_idx), len(classes)), dtype=np.float32)
    ranker_test_probability = np.zeros((len(test_meta), len(classes)), dtype=np.float32)
    ranker_temperatures = {}
    ranker_es_mask = np.isin(valid_idx, es_idx)
    for name, schema in ranker_schemas.items():
        ranker_valid_scores = raw_scores(ranker_data, schema, ranker_models[name], valid_idx)
        ranker_test_scores = raw_scores(ranker_data, schema, ranker_models[name], np.arange(len(test_meta)), test=True)
        ranker_t = ranker_temperature(ranker_valid_scores[ranker_es_mask], ranker_data.y[valid_idx][ranker_es_mask], classes)
        ranker_temperatures[name] = ranker_t
        if RANKER_SOURCE == 'saved':
            assert np.isclose(ranker_t, ranker_bundle['temperatures'][name], rtol=0, atol=1e-8)
        ranker_valid_probability += 0.5 * probabilities(ranker_valid_scores, ranker_t)
        ranker_test_probability += 0.5 * probabilities(ranker_test_scores, ranker_t)
    print('Температуры Ranker:', ranker_temperatures)
    for ranker_probability in [ranker_valid_probability, ranker_test_probability]:
        assert np.isfinite(ranker_probability).all() and (ranker_probability >= 0).all()
        assert np.allclose(ranker_probability.sum(axis=1), 1, atol=1e-5)
    ranker_outer_positions = pd.Index(valid_idx).get_indexer(outer_idx)
    assert (ranker_outer_positions >= 0).all()
    print('Ranker outer Accuracy:', accuracy_score(y[outer_idx], ranker_valid_probability[ranker_outer_positions].argmax(1)))

    # Bridge нужен только для сравнения, а не для обучения и получения вероятностей.
    ranker_bridge_candidates = sorted(INPUT_ROOT.rglob('gru_ranker_bridge.npz'))
    ranker_bridge_matches = [p for p in ranker_bridge_candidates
        if sha256_file(p) == '3adbc897444ad3a458e9fb3477baa2fbdc53c92768316e472e3ec5f9cdf4b8a5']
    if ranker_bridge_matches:
        with np.load(ranker_bridge_matches[0], allow_pickle=False) as bridge:
            assert np.array_equal(bridge['classes'], classes)
            assert np.array_equal(bridge['train_ids'], train_meta.example_id.to_numpy(dtype=str))
            assert np.array_equal(bridge['valid_ids'], train_meta.iloc[valid_idx].example_id.to_numpy(dtype=str))
            assert np.array_equal(bridge['test_ids'], test_meta.example_id.to_numpy(dtype=str))
            for key, rows in [('fit_idx', train_idx), ('valid_idx', valid_idx), ('es_idx', es_idx), ('outer_idx', outer_idx)]:
                assert np.array_equal(bridge[key], rows)
            ranker_bridge_equal = (np.allclose(ranker_valid_probability, bridge['valid_proba'], rtol=0, atol=1e-7)
                                   and np.allclose(ranker_test_probability, bridge['test_proba'], rtol=0, atol=1e-7))
            print('Вероятности совпадают с историческим bridge:', ranker_bridge_equal)
            if RANKER_SOURCE == 'saved':
                assert ranker_bridge_equal, 'Проверка исторического инференса не прошла.'
    else:
        print('Исторического bridge нет: сравнение с ним не выполнено.')


### Выбор температуры и веса GRU

Повторяем код эталона: температуру выбираем по ES log loss из 0.7, 1.0, 1.5 и 2.0, затем вес GRU — по ES Accuracy из 0, 0.25, 0.5, 0.75 и 1. При равенстве берётся первый вариант. Outer не выбирает параметры: как в оригинале, смешанный CSV сохраняем только для двух ненулевых весов и качества не ниже Ranker на outer.

Сохранённый вывод Kaggle подтверждает вес GRU 0.5: ES Accuracy 0.5368, outer Accuracy 0.532631. Это исторические значения, а не результаты нового запуска; вес и температуру вручную не фиксируем. `blend_config.json` больше не является входной зависимостью.

Bridge — архив вероятностей смеси Ranker с ID, классами, разбиением и хешами исходных файлов. Его вероятности побайтово совпадают с `blend_50_50_activity_profile_frozen_main.npz`; метаданные указывают на `selected_ensemble.joblib`. Сам скрипт упаковки bridge не найден, но обучение и получение вероятностей Ranker доступны выше независимо от него и от исторического кэша.


In [ ]:
if RUN_LGBM_ENSEMBLE:
    rank_valid = rank_test = None
    blend_bridge_path = None
    if SMOKE_TEST:
        print('Smoke test: ансамбль и CSV не создаём.')
    elif RANKER_SOURCE == 'bridge':
        bridges = sorted(INPUT_ROOT.rglob('gru_ranker_bridge.npz'))
        if not bridges:
            print('Bridge не подключён: готов самостоятельный submission_gru.csv.')
        else:
            assert len(bridges) == 1, 'Подключите только один проверенный bridge'
            blend_bridge_path = bridges[0]
            assert sha256_file(blend_bridge_path) == '3adbc897444ad3a458e9fb3477baa2fbdc53c92768316e472e3ec5f9cdf4b8a5'
            with np.load(blend_bridge_path, allow_pickle=False) as bridge:
                for key, filename in [('train_sha256', 'train.parquet'), ('test_sha256', 'test.parquet'),
                                      ('dictionary_sha256', 'event_dictionary.parquet')]:
                    assert str(bridge[key].item()) == source_hashes[filename]
                assert np.array_equal(bridge['train_ids'], train_meta.example_id.to_numpy(dtype=str))
                for key, rows in [('fit_idx', train_idx), ('valid_idx', valid_idx), ('es_idx', es_idx), ('outer_idx', outer_idx)]:
                    assert np.array_equal(bridge[key], rows)
                assert len(bridge['classes']) == len(classes) and set(bridge['classes']) == set(classes)
                class_order = pd.Index(bridge['classes']).get_indexer(classes)
                vi = pd.Index(bridge['valid_ids']); ti = pd.Index(bridge['test_ids'])
                assert vi.is_unique and ti.is_unique
                vr = vi.get_indexer(train_meta.iloc[valid_idx].example_id.astype(str))
                tr = ti.get_indexer(test_meta.example_id.astype(str))
                assert (vr >= 0).all() and (tr >= 0).all()
                assert len(vi) == len(valid_idx) and len(ti) == len(test_meta)
                rank_valid = bridge['valid_proba'][vr][:, class_order]
                rank_test = bridge['test_proba'][tr][:, class_order]
    else:
        rank_valid = ranker_valid_probability
        rank_test = ranker_test_probability

    if rank_valid is not None:
        assert valid_probability.shape == rank_valid.shape == (len(valid_idx), len(classes))
        assert test_probability.shape == rank_test.shape == (len(test_meta), len(classes))
        for probability in [rank_valid, rank_test, valid_probability, test_probability]:
            assert np.isfinite(probability).all() and (probability >= 0).all()
            assert np.allclose(probability.sum(1), 1, atol=1e-5)
        es_pos = np.asarray([valid_position[int(row)] for row in es_idx])
        def temper(probability,temperature):
            value = np.log(np.maximum(probability,1e-12))/temperature
            value = np.exp(value-value.max(axis=1,keepdims=True))
            return value/value.sum(axis=1,keepdims=True)
        temperature = min([0.7,1.0,1.5,2.0],key=lambda t: -np.log(np.maximum(
            temper(valid_probability[es_pos],t)[np.arange(len(es_idx)),y[es_idx]],1e-12)).mean())
        calibrated_valid = temper(valid_probability,temperature)
        weights = [0.0,0.25,0.5,0.75,1.0]
        selected_weight = max(weights,key=lambda w: accuracy_score(y[es_idx],
            ((1-w)*rank_valid[es_pos]+w*calibrated_valid[es_pos]).argmax(1)))
        comparison=[]
        for weight in weights:
            probability = (1-weight)*rank_valid+weight*calibrated_valid
            comparison.append(dict(Вес_GRU=weight,Accuracy_ES=accuracy_score(y[es_idx],probability[es_pos].argmax(1)),
                Accuracy_outer=accuracy_score(y[outer_idx],probability[outer_pos].argmax(1)),Выбран_по_ES=weight==selected_weight))
        display(pd.DataFrame(comparison))
        selected = next(row for row in comparison if row['Вес_GRU']==selected_weight)
        if 0<selected_weight<1 and selected['Accuracy_outer']>=comparison[0]['Accuracy_outer']:
            blend_test = (1-selected_weight)*rank_test+selected_weight*temper(test_probability,temperature)
            # Эталон сравниваем только после выбора параметров, без подгонки.
            blend_reference_path = SUBMISSION_DIR / 'submission_gru_lgbm_blend.csv'
            if blend_reference_path.is_file():
                blend_reference = pd.read_csv(blend_reference_path, dtype={'example_id': str})
                assert list(blend_reference.columns) == ['example_id', 'encoded_product']
                assert len(blend_reference) == len(test_meta) and blend_reference.example_id.is_unique
                assert not blend_reference.isna().any().any()
                assert set(blend_reference.example_id) == set(test_meta.example_id.astype(str))
                assert blend_reference.encoded_product.isin(classes).all()
                blend_reference_labels = blend_reference.set_index('example_id').loc[
                    test_meta.example_id.astype(str), 'encoded_product'].to_numpy()
                blend_matches = classes[blend_test.argmax(1)] == blend_reference_labels
                print('Совпадение с существующим CSV:', int(blend_matches.sum()), 'из', len(blend_matches))
            else:
                print('Эталонного CSV нет: историческое совпадение не проверено.')
            blend_submission = save_submission(blend_test,'submission_gru_lgbm_blend.csv')
            blend_config_path = WORK / 'blend_config.json'
            blend_config_suffix = 1
            while blend_config_path.exists():
                blend_config_path = WORK / f'blend_config_{blend_config_suffix}.json'
                blend_config_suffix += 1
            with blend_config_path.open('x') as stream:
                json.dump(dict(gru_weight=selected_weight, temperature=temperature,
                               signature=run_signature, ranker_source=RANKER_SOURCE,
                               ranker_bridge_sha256=sha256_file(blend_bridge_path) if blend_bridge_path else None),
                          stream, indent=2)
        else:
            print('Оснований для отдельного CSV ансамбля пока нет.')


## 10. Вывод

GRU объединяет последовательность событий с агрегированными признаками истории и продуктов. Подтверждённые результаты исторических отправок:

| Модель | Kaggle Public Score |
|---|---:|
| GRU | 0.41203 |
| GRU + LightGBM | 0.40216 |

Самостоятельная GRU показала лучший результат и выбрана итоговым решением. На private её Accuracy составила **0.36692**.

Для запуска нужны три Parquet-файла соревнования, `sample_submission.csv`, установленные зависимости и CUDA GPU. Кэши, сохранённые модели, bridge и входной `blend_config.json` не обязательны. В разделе 9 температура и вес GRU выбираются на ES по исходной процедуре.

`submission_gru.csv` сохраняется после обучения GRU. `submission_gru_lgbm_blend.csv` создаётся при исходном условии: выбраны две ненулевые доли и Accuracy на outer не ниже Ranker. На Kaggle CSV сохраняются в `/kaggle/working/`, локально — в `submits/`; существующие файлы сохраняются, новые получают числовой суффикс.

Статические и выборочные проверки пройдены. Полный пересчёт признаков, обучение и точное совпадение новых предсказаний с историческими CSV пока не подтверждены полным запуском.
